# 외식배달비 조사 자동화

BlueStacks(배달의민족)에서 조사자료를 수집하고 Excel에 기록합니다.

## 조사 흐름

| 순서 | 내용 |
|---:|---|
| 1 | 조사 월·파일 경로 설정 |
| 2 | BlueStacks 연결·Excel 불러오기 |
| 3 | 대상처 검색·배달비·메뉴·사업자정보 수집 함수 |
| 4 | 비교·조사로그 저장·재시작 함수 |
| 5 | 전체조사 또는 선택 지역 조사 실행 |
| 6 | 사람 확인 대상 검토·필요 시 지정 재조사 |
| 부록 | 오류 발생 시 화면 확인 |

## 사용 방법 및 저장 파일

- **전체조사:** `5-3` 실행 → `*_자동전체조사.xlsx`에 주소지마다 저장
- **선택 지역:** `5-4`에서 지역 지정·실행 → 저장 셀 실행 시 `OUTPUT_FILE`에 기록
- **재조사:** `6-2`에서 `TEST_ROWS` 설정 → `6-3`에서 저장

새 조사 초기화가 필요할 때만 `NEW_SURVEY = True`로 설정합니다.

# 1. 조사 설정 ⭐

월이 바뀔 때 수정하는 부분입니다.

## 1-1. 조사 월

In [ ]:
SURVEY_MONTH = 10

## 1-2. 조사 파일

In [ ]:
FILE = r"survey_input.xlsx" #실제 조사 값이 입력되어 있는 원본 파일의 경로를 입력해주세요. (조사 대상처가 들어 있는 입력 Excel)
OUTPUT_FILE = r"survey_output.xlsx" #input 파일과 동일 경로의 output 저장명을 지정하여 입력해주세요. (자동조사 결과를 저장할 Excel)

# 새로운 조사 시작시 True로 변경해주세요.
NEW_SURVEY = False

## 1-3. 조사 주소지

주소지는 매 주소 조사 때마다 바꾸므로 **5. 실제 조사 실행** 맨 위에서 입력합니다.

# 2. 최초 실행 설정

Python을 다시 시작했거나 조사 파일을 바꾼 경우 위에서 아래로 다시 실행합니다.

## 2-1. 라이브러리

In [ ]:
import ast
import hashlib
import io
import json
import os
import re
import subprocess
import sys
import time
import unicodedata
import xml.etree.ElementTree as ET
from collections import defaultdict
from contextlib import redirect_stdout
from copy import copy
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path
from pprint import pprint
from shutil import copy2
import uiautomator2 as u2
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter, column_index_from_string

## 2-2. BlueStacks 연결

처음 연결할 때 Windows 터미널에서 ADB 상태를 확인한 뒤 아래 셀을 실행합니다.

### 먼저 터미널에서

#### > (결과 True 나와야함)

> Test-Path "C:\Program Files\BlueStacks_nxt\HD-Adb.exe"  

> & "C:\Program Files\BlueStacks_nxt\HD-Adb.exe" start-server


> & "C:\Program Files\BlueStacks_nxt\HD-Adb.exe" devices

입력 후

In [ ]:
d = u2.connect()
print(d.info)
print("BlueStacks 화면 크기:", d.window_size())

## 2-3. Excel 불러오기

- **새 조사**: 위 `NEW_SURVEY = True`로 설정한 뒤 이 셀을 실행합니다. `FILE`의 조사자료를 기준으로 `OUTPUT_FILE`에 저장하고 `조사로그` 기록은 0건으로 초기화합니다. 기존 결과 파일이 있으면 **`초기화`를 직접 입력해야만** 덮어쓰며, 먼저 자동 백업합니다. 정상적으로 초기화되면 실행 중 변수는 자동으로 `False`로 바뀝니다.
- **이어서 조사**: `NEW_SURVEY = False`이면 기존 `OUTPUT_FILE`의 조사자료·로그를 그대로 불러옵니다.
- **주의**: 새 조사 설정을 다시 실행하면 `NEW_SURVEY` 값이 셀에 적힌 기본값(`False`)으로 돌아옵니다. 다음 새 조사 때에만 다시 `True`로 변경하세요.


In [ ]:
def get_month_cols(month):
    start = (column_index_from_string("BC") + (month - 8) * 6)
    return {
        "store_min": get_column_letter(start),
        "store_fee": get_column_letter(start + 1),
        "saver_min": get_column_letter(start + 2),
        "saver_fee": get_column_letter(start + 3),
        "one_min": get_column_letter(start + 4),
        "one_fee": get_column_letter(start + 5)
    }

def get_result_cols(month):
    start = (column_index_from_string("BC") + (month - 8) * 6 + 6)
    return {
        "change": get_column_letter(start),
        "store_status": get_column_letter(start + 1),
        "saver_status": get_column_letter(start + 2),
        "one_status": get_column_letter(start + 3),
        "notes": get_column_letter(start + 4)
    }

In [ ]:
SHEET_NAME = "조사자료(rawdata)"
PREV_COLS = get_month_cols(SURVEY_MONTH - 1)
CURR_COLS = get_month_cols(SURVEY_MONTH)
RESULT_COLS = get_result_cols(SURVEY_MONTH)
if Path(FILE).resolve() == Path(OUTPUT_FILE).resolve():
    raise ValueError("FILE과 OUTPUT_FILE은 서로 다른 파일이어야 합니다.")
output_exists = Path(OUTPUT_FILE).exists()
if NEW_SURVEY or not output_exists:
    if output_exists:
        answer = input("새 조사 시작: 기존 OUTPUT_FILE의 조사내용도 FILE 기준으로 바뀝니다. " "기존 결과는 자동 백업됩니다. 진행하려면 '초기화' 입력: ")
        if answer.strip() != "초기화":
            NEW_SURVEY = False
            raise RuntimeError("새 조사 초기화를 취소했습니다. 기존 파일은 유지됩니다.")
        backup_path = Path(OUTPUT_FILE).with_name(
            Path(OUTPUT_FILE).stem
            + "_초기화전백업_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            + Path(OUTPUT_FILE).suffix
        )
        copy2(OUTPUT_FILE, backup_path)
        print("기존 결과 파일 백업:", backup_path)
    wb = load_workbook(FILE)
    if "조사로그" in wb.sheetnames:
        log_ws = wb["조사로그"]
        if log_ws.max_row > 1:
            log_ws.delete_rows(2, log_ws.max_row - 1)
        if log_ws.auto_filter.ref:
            log_ws.auto_filter.ref = f"A1:{get_column_letter(log_ws.max_column)}1"
    else:
        log_ws = wb.create_sheet("조사로그")
        log_ws["A1"] = "기록일"
    wb.save(OUTPUT_FILE)
    NEW_SURVEY = False
    print("새 조사 시작 - 조사로그 0건:", OUTPUT_FILE)
else:
    wb = load_workbook(OUTPUT_FILE)
    print("기존 조사 이어하기 - 조사로그 유지:", OUTPUT_FILE)
ws = wb[SHEET_NAME]
print("조사월:", SURVEY_MONTH)
print("전월 열:", PREV_COLS)
print("당월 열:", CURR_COLS)
print("결과 열:", RESULT_COLS)
print("파일 불러오기 완료:", OUTPUT_FILE)

## 2-4. 조사대상 생성

Excel의 H열 주소지별로 조사대상을 묶습니다. 파일을 바꿨다면 반드시 이 셀도 다시 실행합니다.

In [ ]:
groups = defaultdict(list)
region_groups = defaultdict(lambda: defaultdict(list))
EXCLUDED_REGIONS = {"전국", "중앙"}
for row in range(8, ws.max_row + 1):
    region = ws[f"F{row}"].value
    address = ws[f"H{row}"].value
    store_name = ws[f"I{row}"].value
    if not address or not store_name:
        continue
    store_data = {
        "row": row,
        "category": ws[f"D{row}"].value,
        "item": ws[f"E{row}"].value,
        "store": store_name,
        "owner": ws[f"J{row}"].value,
        "business_address": ws[f"K{row}"].value,
        "menu": ws[f"L{row}"].value
    }
    groups[address].append(store_data)
    if region:
        region = str(region).strip()
        if region not in EXCLUDED_REGIONS:
            region_groups[region][address].append(store_data)
gyeonggi = list(region_groups["경기"].items())
mid = (len(gyeonggi) + 1) // 2
region_groups["경기1"] = dict(gyeonggi[:mid])
region_groups["경기2"] = dict(gyeonggi[mid:])
for region in ["경기1", "경기2"]:
    addresses = region_groups[region]
    count = sum(len(stores) for stores in addresses.values())
    print(f"{region}: 주소지 {len(addresses)}개 / 대상처 {count}개")

# 3. 자동화 기본 함수



## 3-1. 화면 이동

In [ ]:
def go_home():
    package = d.app_current()["package"]
    for _ in range(4):
        if "현재 위치를 변경할 수 있습니다"in d.dump_hierarchy():
            return True
        d.press("back")
        time.sleep(1)
        if d.app_current()["package"] != package:
            d.app_start(package)
            time.sleep(3)
    return "현재 위치를 변경할 수 있습니다." in d.dump_hierarchy()

In [ ]:
def set_address(address):

    # 1. 현재 주소 선택
    current_address = d.xpath('//*[contains(@content-desc,"현재 위치를 변경할 수 있습니다")]')
    if not current_address.wait(timeout=5):
        print("현재 주소 버튼 못 찾음")
        return False
    current_address.click()
    address_page = d.xpath('//*[contains(@text,"도로명") or contains(@content-desc,"도로명")]')
    if not address_page.wait(timeout=5):
        print("주소 설정 화면 진입 실패")
        return False

    # 2. 주소 검색창 클릭
    root = ET.fromstring(d.dump_hierarchy())
    for node in root.iter():
        text = node.attrib.get("text", "")
        desc = node.attrib.get("content-desc", "")
        if "도로명" in text or "도로명" in desc:
            x1, y1, x2, y2 = map(int, re.findall(r"\d+", node.attrib["bounds"]))
            d.click((x1+x2)//2, (y1+y2)//2)
            search_bottom = y2
            break

    # 3. 주소 입력
    box = d(focused=True)
    if not box.wait(timeout=5):
        print("주소 입력창 못 찾음")
        return False
    box.set_text(address)
    d.press("enter")

    # 4. 검색 결과 확인
    parts = [re.sub(r"[,]", "", part).strip() for part in address.split()]
    road = next((part for part in parts if part.endswith(("대로", "로", "길"))), None)
    if not road:
        print("도로명 추출 실패:", address)
        return False
    for _ in range(30):
        root = ET.fromstring(d.dump_hierarchy())
        results = []
        for node in root.iter():
            text = node.attrib.get("text", "")
            desc = node.attrib.get("content-desc", "")
            if road in text or road in desc:
                x1, y1, x2, y2 = map(int, re.findall(r"\d+", node.attrib["bounds"]))
                if y1 > search_bottom + 100:
                    results.append((y1, x1, x2, y2))
        if results:
            y1, x1, x2, y2 = min(results)
            d.click((x1+x2)//2, (y1+y2)//2)
            break
        time.sleep(0.3)
    else:
        print("주소 검색결과 못 찾음")
        return False

    # 5. 상세주소 입력
    box = d(focused=True)
    if not box.wait(timeout=5):
        print("상세주소 입력창 못 찾음")
        return False
    box.set_text("정문")

    # 6. 주소 등록
    root = ET.fromstring(d.dump_hierarchy())
    for node in root.iter():
        if node.attrib.get("text", "") == "주소 등록":
            x1, y1, x2, y2 = map(int, re.findall(r"\d+", node.attrib["bounds"]))
            d.click((x1+x2)//2, (y1+y2)//2)
            break

    # 7. 팝업 저장
    for _ in range(30):
        xml = d.dump_hierarchy()
        if "주소는 10개까지" in xml:
            d.click(575,923)
            print("저장 클릭")
            break
        time.sleep(0.3)
    else:
        print("주소 10개 팝업 못 찾음")
        return False

    # 8. 홈 화면 확인
    home = d.xpath('//*[@clickable="true" and' '(contains(@text,"검색")or contains(@content-desc,"검색"))]')
    if not home.wait(timeout=20):
        print("배민 첫 화면 로딩 실패")
        return False
    print("주소 설정 완료:", address)
    return True

## 3-2. 대상처 검색

긴 `open_store()` 안에 있던 보조 함수를 `def` 단위로 분리했습니다.  
**아래 셀들을 위에서 아래 순서로 실행한 뒤 마지막 `open_store()`를 실행하면 됩니다.**

### 검색결과 이름 정리

In [ ]:
def clean_result_name(name):
    name = str(name).strip()
    name = re.sub(r"^민트스타\s*가게,\s*", "", name)
    name = re.sub(r",\s*준비중\s*$", "", name)
    name = re.sub(r",\s*배달팁\s*(?:[\d,]+\s*원|할인)\s*$", "", name)
    name = name.replace("＆", "&")
    return name.strip()

### 지점명 분리

In [ ]:
def split_store_name(name):
    generic_tails = {"본점", "직영점", "가맹점", "직영"}
    name = clean_result_name(name)
    tokens = name.split()
    while (tokens and tokens[-1] in generic_tails):
        tokens.pop()
    branch = None
    if tokens:
        last = tokens[-1]
        if (last.endswith("점") and len(last) >= 2):
            branch = (last .replace("직영점", "점") .replace("본점", "점"))
            tokens = tokens[:-1]
    core = "".join(tokens).lower()
    core = re.sub(r"[^가-힣a-z0-9]", "", core)
    return core, branch

### 기존 자동매칭

In [ ]:
def name_match(target_name, candidate_name):
    if "검색결과 더보기" in candidate_name:
        return False
    candidate_name = clean_result_name(candidate_name)
    candidate_clean = (candidate_name .replace(" ", "") .replace("직영", "") .lower())
    target_clean = (target_name .replace(" ", "") .replace("직영", "") .lower())
    if (target_clean in candidate_clean or candidate_clean in target_clean):
        return True
    words = target_name.split()
    for word in words:
        word = (word .replace(" ", "") .replace("직영", "") .lower())
        if not word:
            continue
        if word not in candidate_clean:
            return False
    return True

### 동일 점포 가능성 판단

In [ ]:
def same_store_candidate(target_name, candidate_name, menu_name):
    def clean_brand_token(token):
        token = str(token or "").replace("직영", "").lower()
        return re.sub(r"[^가-힣a-z0-9]", "", token)
    target_name = clean_result_name(target_name)
    candidate_name = clean_result_name(candidate_name)
    target_tokens = target_name.split()
    candidate_tokens = candidate_name.split()
    if not target_tokens or not candidate_tokens:
        return False
    target_brand = clean_brand_token(target_tokens[0])
    candidate_brand = clean_brand_token(candidate_tokens[0])
    if not target_brand or not candidate_brand:
        return False
    if target_brand == candidate_brand:
        return True
    if min(len(target_brand), len(candidate_brand)) < 3:
        return False
    if target_brand in candidate_brand or candidate_brand in target_brand:
        return True
    common_prefix_len = 0
    for target_char, candidate_char in zip(target_brand, candidate_brand):
        if target_char != candidate_char:
            break
        common_prefix_len += 1
    if common_prefix_len >= 3:
        prefix_ratio = common_prefix_len / min(len(target_brand), len(candidate_brand))
        if prefix_ratio >= 0.4:
            return True
    similarity = SequenceMatcher(None, target_brand, candidate_brand).ratio()
    if similarity >= 0.80:
        return True
    return False

### 대상처 검색 실행

In [ ]:
def open_store(store_name, item_name=None):
    search = d.xpath('//*[@clickable="true" and (contains(@text,"검색") or contains(@content-desc,"검색"))]')
    if not search.wait(timeout=5):
        print("검색창 못 찾음")
        return {"status": "TECHNICAL_FAIL", "reason": "검색창 못 찾음", "candidates": []}
    search.click()
    search_names = [store_name]
    alt_name = store_name.replace("직영", "")
    if alt_name != store_name:
        search_names.append(alt_name)
    same_store_candidates = []
    saw_store_card = False
    saw_no_result_message = False

    # 1. 대상처 검색
    for search_name in search_names:
        print("검색 시도:", search_name)
        box = d(className="android.widget.EditText")
        if not box.wait(timeout=5):
            print("검색 입력창 못 찾음")
            return {"status": "TECHNICAL_FAIL", "reason": "검색 입력창 못 찾음", "candidates": []}
        box.set_text(search_name)
        time.sleep(0.5)
        d.press("enter")
        time.sleep(0.5)

        # 2. 검색결과 확인
        for _ in range(20):
            xml = d.dump_hierarchy()
            root = ET.fromstring(xml)
            if any(phrase in xml for phrase in [
                "검색 결과가 없어요",
                "검색결과가 없어요",
                "검색 결과가 없습니다",
                "검색결과가 없습니다",
                "검색 결과 없음"
            ]):
                saw_no_result_message = True
            results = []
            screen_width, screen_height = d.window_size()
            delivery_end_y = screen_height + 1

            # 3. 배달 영역 끝 위치 확인
            for node in root.iter():
                text = node.attrib.get("text", "").strip()
                desc = node.attrib.get("content-desc", "").strip()
                value = text or desc
                bounds = get_bounds(node)
                if not value or not bounds:
                    continue
                x1, y1, x2, y2 = bounds
                if y1 > 250 and x1 == 0 and x2 >= screen_width * 0.9 and (value == "픽업" or "픽업 이런 가게는 어때요?" in value):
                    delivery_end_y = min(delivery_end_y, y1)

            # 4. 배달 영역 가게 수집
            for node in root.iter():
                text = node.attrib.get("text", "").strip()
                desc = node.attrib.get("content-desc", "").strip()
                value = text or desc
                if not value:
                    continue
                bounds = get_bounds(node)
                if not bounds:
                    continue
                x1, y1, x2, y2 = bounds
                if not (y1 > 300
                        and y1 < delivery_end_y
                        and x1 < 100
                        and "검색결과 더보기" not in value
                        and (y2 - y1) > 100):
                    continue
                saw_store_card = True
                candidate = clean_result_name(value)
                search_key = re.sub(r"[^가-힣a-z0-9]", "", str(search_name).lower())
                candidate_key = re.sub(r"[^가-힣a-z0-9]", "", str(candidate).lower())
                exact_match = name_match(search_name, value)
                punctuation_match = bool(search_key and candidate_key and search_key == candidate_key)
                if exact_match or punctuation_match:
                    results.append({"name": value,
                                    "clean_name": candidate,
                                    "bounds": (x1, y1, x2, y2),
                                    "name_typo": punctuation_match and not exact_match,
                                    "searched_name": search_name})
                elif same_store_candidate(store_name, candidate, item_name):
                    if candidate not in same_store_candidates:
                        same_store_candidates.append(candidate)

            # 5. 자동매칭 결과 처리
            if results:
                unique_results = []
                seen = set()
                for r in results:
                    name_key = r["clean_name"].replace(" ", "").replace("직영", "").lower()
                    key = (name_key, r["bounds"])
                    if key not in seen:
                        seen.add(key)
                        unique_results.append(r)
                print("검색결과 개수:", len(unique_results))
                for r in unique_results:
                    print(r["clean_name"], r["bounds"])
                if len(unique_results) >= 2:
                    print("사람확인 필요: 대상처 매칭 후보 여러 개")
                    return {"status": "MULTIPLE_MATCH_REVIEW",
                            "reason": "대상처 매칭 후보 여러 개",
                            "candidates": [r["clean_name"] for r in unique_results]}
                selected = unique_results[0]
                x1, y1, x2, y2 = selected["bounds"]
                if selected.get("name_typo"):
                    print("※ 대상처명 표기 차이 자동통과")
                    print("엑셀:", selected["searched_name"])
                    print("배민:", selected["clean_name"])
                d.click((x1 + x2) // 2, (y1 + y2) // 2)
                print("대상처 클릭:", selected["clean_name"])

                # 6. 상세화면 진입 확인
                for _ in range(30):
                    detail_xml = d.dump_hierarchy()
                    if "함께주문" in detail_xml:
                        print("가게 상세화면 진입 완료")
                        return {"status": "FOUND",
                                "reason": "",
                                "selected": selected["clean_name"],
                                "name_typo": selected.get("name_typo", False),
                                "searched_name": selected.get("searched_name"),
                                "candidates": []}
                    time.sleep(0.3)
                print("가게 상세화면 진입 실패")
                return {"status": "TECHNICAL_FAIL", "reason": "가게 상세화면 진입 실패", "candidates": [selected["clean_name"]]}
            time.sleep(0.3)

    # 7. 자동매칭 없음
    if same_store_candidates:
        print("동일 대상처 가능 후보:")
        for name in same_store_candidates:
            print("-", name)
        return {"status": "SAME_STORE_REVIEW", "reason": "동일 대상처 가능 후보", "candidates": same_store_candidates}
    if saw_store_card or saw_no_result_message:
        print("기존 대상처 없음 -> 신규 대상처 탐색 필요")
        return {"status": "REPLACEMENT_NEEDED", "reason": "신규 대상처 탐색 필요", "candidates": []}
    print("검색결과 화면 확인 실패")
    return {"status": "TECHNICAL_FAIL", "reason": "검색결과 화면 확인 실패", "candidates": []}

## 3-3. 배달정보 조사

In [ ]:
def to_number(value):
    if not value:
        return None
    nums = re.findall(r"[\d,]+", str(value))
    if nums:
        return int(nums[-1].replace(",", ""))
    return None

In [ ]:
def get_delivery():
    root = ET.fromstring(d.dump_hierarchy())

    # 1. 배달정보 펼치기
    for node in root.iter():
        desc = node.attrib.get("content-desc", "")
        if "배달유형별 배달팁" in desc:
            if "펼치기" in desc:
                x1, y1, x2, y2 = map(int, re.findall(r"\d+", node.attrib["bounds"]))
                d.click((x1 + x2) // 2, (y1 + y2) // 2)
                time.sleep(0.5)
                root = ET.fromstring(d.dump_hierarchy())
            break

    # 2. 가게 배달정보 수집
    items = []
    reading = False
    for node in root.iter():
        text = node.attrib.get("text", "").strip()
        desc = node.attrib.get("content-desc", "").strip()
        if text == "최소주문":
            reading = True
        if text == "지금 배달되는 비슷한 가게":
            break
        if reading and (text or desc):
            items.append((text, desc))

    # 3. 최소주문금액
    min_order = None
    for i, (text, desc) in enumerate(items):
        if text == "최소주문":
            for text2, desc2 in items[i+1:i+5]:
                if text2.endswith("원"):
                    min_order = to_number(text2)
                    break
            break

    # 4. 배달종류별 배달비
    def find_fee(delivery_type):
        for i, (text, desc) in enumerate(items):
            if text == delivery_type or desc == delivery_type:
                for text2, desc2 in items[i+1:i+6]:
                    if (
                        text2 in ["가게배달", "알뜰배달", "한집배달"]
                        or desc2 in ["가게배달", "알뜰배달", "한집배달"]
                    ):
                        break
                    if text2.startswith("배달팁"):
                        return to_number(text2)
                    if text2.endswith("원") and "정가" not in desc2:
                        return to_number(text2)
                return None
        return None
    result = {
        "min_order": min_order,
        "store_fee": find_fee("가게배달"),
        "saver_fee": find_fee("알뜰배달"),
        "one_fee": find_fee("한집배달")
    }
    print(result)
    return result

## 3-4. 사업자정보 조사

In [ ]:
def get_information():
    print("가게정보 클릭")
    d.click(771, 480)
    time.sleep(2)

    # 1. 가게 주소 수집
    root = ET.fromstring(d.dump_hierarchy())
    top_texts = []
    for node in root.iter():
        text = node.attrib.get("text", "").strip()
        if text:
            top_texts.append(text)
    store_address = None
    for i, text in enumerate(top_texts):
        if text == "주소" and i + 1 < len(top_texts):
            store_address = top_texts[i + 1]
            break
    print("가게 주소:", store_address)

    # 2. 사업자정보 수집
    for i in range(5):
        xml = d.dump_hierarchy()
        if "상호명" in xml and "사업자주소" in xml:
            print("사업자정보 찾음")
            break
        print("스크롤:", i + 1)
        d.swipe(450, 1200, 450, 450, duration=0.4)
        time.sleep(0.4)
    else:
        print("사업자정보 못 찾음")
        return None
    root = ET.fromstring(d.dump_hierarchy())
    texts = []
    for node in root.iter():
        text = node.attrib.get("text", "").strip()
        if text:
            texts.append(text)
    def get_value(label):
        for i, text in enumerate(texts):
            if text == label and i + 1 < len(texts):
                return texts[i + 1]
        return None
    owner = get_value("대표자명")
    business_name = get_value("상호명")
    business_address = get_value("사업자주소")
    if business_address:
        business_address = business_address.replace(
            "위 주소는 사업자등록증에 표기된 정보입니다.",
            ""
        ).strip()
    result = {"owner": owner, "business_name": business_name, "address": business_address, "store_address": store_address}
    print(result)
    return result

## 3-5. 메뉴 조사

In [ ]:
def get_texts():
    xml = d.dump_hierarchy()
    root = ET.fromstring(xml)
    texts = []
    for node in root.iter():
        text = node.attrib.get("text","").strip()
        if text:
            texts.append(text)
    return texts

In [ ]:
def open_menu_search(store_name):
    for attempt in range(2):
        xml = d.dump_hierarchy()
        root = ET.fromstring(xml)
        nodes = []
        for node in root.iter():
            text = node.attrib.get("text", "").strip()
            bounds = node.attrib.get("bounds", "")
            nums = list(map(int, re.findall(r"\d+", bounds)))
            if len(nums) != 4:
                continue
            x1, y1, x2, y2 = nums
            nodes.append({"text": text, "bounds": (x1, y1, x2, y2)})
        target = None

        # 1. 대상처명 검색 메뉴
        search_titles = {f"{store_name}로 검색한 메뉴", f"{store_name}으로 검색한 메뉴"}
        for item in nodes:
            if item["text"] in search_titles:
                target = item
                print("대상처 검색 메뉴 기준 찾음:", item["text"])
                break

        # 2. 주요 메뉴 섹션
        if target is None:
            for item in nodes:
                if item["text"] == "배민Only 메뉴":
                    target = item
                    print("메뉴 기준 찾음:", item["text"])
                    break
        if target is None:
            for item in nodes:
                if item["text"] == "가장 인기 있는 메뉴":
                    target = item
                    print("메뉴 기준 찾음:", item["text"])
                    break
        if target is None:
            for item in nodes:
                if item["text"] == "메뉴소개":
                    target = item
                    print("메뉴 기준 찾음:", item["text"])
                    break

        # 3. 가게별 메뉴 섹션 제목
        if target is None:
            for item in nodes:
                text = item["text"]
                if not text:
                    continue
                if text in ("최근 주문 많은 조합", "추천 조합", "인기 조합"):
                    continue
                x1, y1, x2, y2 = item["bounds"]
                if 28 <= x1 <= 35 and y1 >= 450 and 35 <= (y2 - y1) <= 55 and not re.match(r"^(인기\s*\d+위|사장님 추천|리뷰\s*\d+|품절됐어요)$", text):
                    target = item
                    print("메뉴 섹션 기준 찾음:", text)
                    break

        # 4. [[...]] 메뉴 섹션
        if target is None:
            for item in nodes:
                text = item["text"]
                if text.startswith("[[") and text.endswith("]]"):
                    target = item
                    print("메뉴 섹션 기준 찾음:", text)
                    break

        # 5. 검색 돋보기 클릭
        if target is not None:
            x1, y1, x2, y2 = target["bounds"]
            search_x = x1 + 26
            search_y = y1 - 64
            print("메뉴 기준 위치:", target["bounds"])
            print("돋보기 클릭:", search_x, search_y)
            d.click(search_x, search_y)
            time.sleep(0.5)
            return True

        # 6. 한 번 스크롤 후 재탐색
        if attempt == 0:
            print("메뉴 영역 안 보임 → 한 번 스크롤")
            d.swipe(450, 1200, 450, 700, duration=0.4)
            time.sleep(0.5)
    print("메뉴 영역 못 찾음")
    print("\n현재 화면 메뉴 관련 텍스트:")
    for item in nodes:
        text = item["text"]
        if not text:
            continue
        x1, y1, x2, y2 = item["bounds"]
        if y1 >= 400:
            print(text, item["bounds"])
    return False

In [ ]:
def search_menu(menu_name, menu_category=None):
    d.send_keys(menu_name)
    time.sleep(0.5)
    d.send_action("search")
    time.sleep(1)
    texts = get_texts()
    target = (str(menu_name) .replace(" ", "") .lower())
    for text in texts:
        if "\n" in str(text):
            continue
        value = (str(text) .replace(" ", "") .lower())
        if target in value or value in target:
            print("기존 메뉴 있음:", text)
            return True, []
    print("기존 메뉴 검색 안 됨")
    return False, []

In [ ]:
def clean_menu_text(menu):
    text = unicodedata.normalize("NFKC", str(menu or "")).lower()
    return re.sub(r"[^가-힣a-z0-9]", "", text)

In [ ]:
def get_popular_menu_candidates(old_menu):
    texts = get_texts()
    candidates = []
    seen = set()
    old_clean = clean_menu_text(old_menu)
    in_menu_area = False
    expect_menu = False
    for text in texts:
        value = str(text or "").strip()
        value_check = re.sub(r"\s+", " ", value).strip()
        if not value_check:
            continue
        if value_check == "가장 인기 있는 메뉴" or value_check.endswith("로 검색한 메뉴"):
            in_menu_area = True
            expect_menu = False
            continue
        if not in_menu_area:
            continue
        if "한 달간 주문수가" in value_check:
            continue
        if value_check == "사장님 추천" or re.fullmatch(r"인기\s*\d+\s*위", value_check):
            expect_menu = True
            continue
        if not expect_menu:
            continue
        if re.fullmatch(r"\d{1,2}:\d{2}", value_check):
            continue
        if value_check.startswith("리뷰 "):
            continue
        if "품절" in value_check:
            continue
        if value_check.startswith("오늘 ") and "오픈" in value_check:
            continue
        if len(value_check) > 80:
            continue
        menu_name = value
        menu_clean = clean_menu_text(menu_name)
        if not menu_clean:
            continue
        if menu_name in seen:
            expect_menu = False
            continue
        seen.add(menu_name)
        if min(len(old_clean), len(menu_clean)) >= 3 and (old_clean in menu_clean or menu_clean in old_clean):
            similarity = 1.0
        else:
            similarity = SequenceMatcher(None, old_clean, menu_clean).ratio()
        candidates.append({"rank": len(candidates) + 1, "menu": menu_name, "similarity": similarity})
        expect_menu = False
        if len(candidates) == 5:
            break
    recommendation = None
    if candidates:
        best = max(candidates, key=lambda x: x["similarity"])
        if best["similarity"] > 0:
            recommendation = best
    return candidates, recommendation

## 3-6. 신규 대상처 탐색

기존 대상처가 검색되지 않을 때 현재 화면에 보이는 후보를 수집하고 **상호 10 / 메뉴 50 / 배달비 20 / 리뷰 20** 기준으로 점수화합니다.

여기도 내부 보조 함수를 `def` 단위로 분리했습니다.

### 화면 좌표 읽기

In [ ]:
def get_bounds(node):
    nums = re.findall(r"\d+", node.attrib.get("bounds", ""))
    if len(nums) == 4:
        return list(map(int, nums))
    return None

### 상호명 정리

In [ ]:
def brand_name(name):
    name = str(name or "").strip().lower()
    name = name.replace("민트스타 가게,", "").strip()
    name = re.sub(r",?\s*준비중$", "", name).strip()
    parts = name.split()
    if len(parts) >= 2 and parts[-1].endswith("점"):
        parts = parts[:-1]
    return " ".join(parts)

### 신규 대상처 후보 탐색

In [ ]:
def find_replacement_candidates(store):
    old_store = str(store.get("store") or "").strip()
    old_menu = str(store.get("menu") or "").strip()
    if not old_menu:
        return {"status": "TECHNICAL_FAIL", "reason": "기존 메뉴 없음", "candidates": [], "recommendation": None}
    print("\n===== 신규 대상처 탐색 =====")
    print("기존 대상처:", old_store)
    print("검색 메뉴:", old_menu)

    # 1. 기존 메뉴 검색
    search = d.xpath('//*[@clickable="true" and ' '(contains(@text,"검색") or contains(@content-desc,"검색"))]')
    if not search.wait(timeout=5):
        return {"status": "TECHNICAL_FAIL", "reason": "검색창 못 찾음", "candidates": [], "recommendation": None}
    search.click()
    time.sleep(0.5)
    box = d(className="android.widget.EditText")
    if not box.wait(timeout=5):
        return {"status": "TECHNICAL_FAIL", "reason": "검색 입력창 못 찾음", "candidates": [], "recommendation": None}
    box.set_text(old_menu)
    d.press("enter")
    time.sleep(1)

    # 2. 배달 탭
    delivery_ok = False
    delivery_tab_bounds = None
    for attempt in range(3):
        root = ET.fromstring(d.dump_hierarchy())
        delivery_node = None
        for node in root.iter():
            if node.attrib.get("content-desc", "").strip() == "배달 탭":
                delivery_node = node
                break
        if delivery_node is None:
            time.sleep(0.5)
            continue
        bounds = get_bounds(delivery_node)
        if not bounds:
            time.sleep(0.5)
            continue
        delivery_tab_bounds = bounds
        if delivery_node.attrib.get("selected", "false") == "true":
            delivery_ok = True
            break
        x1, y1, x2, y2 = bounds
        print(f"배달 탭 클릭 시도 {attempt + 1}")
        d.click((x1 + x2) // 2, (y1 + y2) // 2)
        time.sleep(1)
        root = ET.fromstring(d.dump_hierarchy())
        for node in root.iter():
            if node.attrib.get("content-desc", "").strip() != "배달 탭":
                continue
            checked_bounds = get_bounds(node)
            if checked_bounds:
                delivery_tab_bounds = checked_bounds
            if node.attrib.get("selected", "false") == "true":
                delivery_ok = True
            break
        if delivery_ok:
            break
    if not delivery_ok or not delivery_tab_bounds:
        print("배달 탭 진입 실패")
        return {"status": "TECHNICAL_FAIL", "reason": "배달 탭 진입 실패", "candidates": [], "recommendation": None}
    print("배달 탭 진입 확인 완료")

    # 3. 검색결과 확인
    root = ET.fromstring(d.dump_hierarchy())
    screen_texts = [node.attrib.get("text", "").strip() for node in root.iter() if node.attrib.get("text", "").strip()]
    no_direct_result = ("검색어와 비슷한 결과를 찾아봤어요." in screen_texts and "이런 메뉴나 가게는 어때요?" in screen_texts)
    if no_direct_result:
        print("대체 대상처 검색결과 없음")
        return {"status": "NO_REPLACEMENT_CANDIDATE", "reason": "대체 대상처 없음", "candidates": [], "recommendation": None}

    # 4. 검색어 보정 안내문 확인
    correction_hint_bounds = None
    correction_hint_text = None
    for node in root.iter():
        text = node.attrib.get("text", "").strip()
        if text.endswith("찾으시는 것 같아요."):
            correction_hint_bounds = get_bounds(node)
            correction_hint_text = text
            break
    if correction_hint_bounds:
        print("검색어 보정 안내문 감지:", correction_hint_text)

    # 5. 주문 많은 순
    sort_bounds = None
    sort_state = None
    for _ in range(20):
        root = ET.fromstring(d.dump_hierarchy())
        for node in root.iter():
            text = node.attrib.get("text", "").strip()
            desc = node.attrib.get("content-desc", "").strip()
            bounds = get_bounds(node)
            if not bounds:
                continue
            if "주문 많은 순" in text or "주문 많은 순" in desc:
                sort_bounds = bounds
                sort_state = "ORDER"
                break
            if text == "기본순" or desc == "기본순":
                sort_bounds = bounds
                sort_state = "DEFAULT"
                break
        if sort_bounds:
            break
        if (any(re.fullmatch(r"배달\s*1", n.attrib.get("text", "").strip()) for n in root.iter())
                and any(n.attrib.get("text", "").strip().startswith("최소주문")
                        for n in root.iter())):
            break
        time.sleep(0.3)
    if not sort_bounds:
        root = ET.fromstring(d.dump_hierarchy())
        single_cards = []
        for node in root.iter():
            desc = node.attrib.get("content-desc", "").strip()
            bounds = get_bounds(node)
            if not desc or not bounds:
                continue
            x1, y1, x2, y2 = bounds
            if x1 >= 100 or not (200 <= y1 < 1500) or y2 - y1 < 90:
                continue
            if any(n.attrib.get("text", "").strip().startswith("최소주문")
                   and (lambda b: bool(b) and y1 <= b[1] <= y2 + 15)(get_bounds(n))
                   for n in root.iter()):
                single_cards.append((desc, bounds))
        if len(single_cards) == 1:
            print("검색 결과 1개 확인 → 정렬 생략")
            sort_state = "SINGLE"
        else:
            print("정렬 기준 및 단일 가게 카드 확인 실패")
            return {"status": "TECHNICAL_FAIL",
                    "reason": "정렬 UI 없음 - 검색 결과 화면 확인 필요",
                    "candidates": [name for name, _ in single_cards],
                    "recommendation": None}
    if sort_state == "SINGLE":
        pass
    elif sort_state == "ORDER":
        print("이미 주문 많은 순 상태")
    else:
        x1, y1, x2, y2 = sort_bounds
        sort_x = (x1 + x2) // 2
        if correction_hint_bounds:
            hx1, hy1, hx2, hy2 = correction_hint_bounds
            sort_y = hy1 - 60
            print("기본순 기준 위치:", sort_bounds)
            print("검색어 보정 안내문 위치:", correction_hint_bounds)
            print("보정 안내문 기준 정렬 버튼 클릭:", (sort_x, sort_y))
        else:
            sort_y = y1 - 22
            print("기본순 기준 위치:", sort_bounds)
            print("정렬 버튼 상대좌표 클릭:", (sort_x, sort_y))
        d.click(sort_x, sort_y)
        order_bounds = None
        for attempt in range(20):
            root = ET.fromstring(d.dump_hierarchy())
            for node in root.iter():
                text = node.attrib.get("text", "").strip()
                desc = node.attrib.get("content-desc", "").strip()
                if "주문 많은 순" in text or "주문 많은 순" in desc:
                    order_bounds = get_bounds(node)
                    if order_bounds:
                        break
            if order_bounds:
                break
            if attempt == 6:
                d.click(sort_x, sort_y)
            time.sleep(0.3)
        if not order_bounds:
            print("주문 많은 순 못 찾음")
            return {"status": "TECHNICAL_FAIL", "reason": "주문 많은 순 못 찾음", "candidates": [], "recommendation": None}
        x1, y1, x2, y2 = order_bounds
        d.click((x1 + x2) // 2, (y1 + y2) // 2)
        time.sleep(1)
        print("주문 많은 순 선택 완료")

    # 6. 후보 수집
    root = ET.fromstring(d.dump_hierarchy())
    old_clean = old_menu.replace(" ", "").lower()
    candidates = []
    for node in root.iter():
        if node.attrib.get("clickable") != "true":
            continue
        bounds = get_bounds(node)
        if not bounds:
            continue
        x1, y1, x2, y2 = bounds
        if not (x1 == 0
                and x2 >= 850
                and 300 <= y1 < 1600
                and (y2 - y1) >= 150):
            continue
        texts = []
        descs = []
        values = []
        for child in node.iter():
            text = child.attrib.get("text", "").strip()
            desc = child.attrib.get("content-desc", "").strip()
            if text:
                if text not in texts:
                    texts.append(text)
                if text not in values:
                    values.append(text)
            if desc:
                if desc not in descs:
                    descs.append(desc)
                if desc not in values:
                    values.append(desc)
        if not any(value.startswith("최소주문") for value in values):
            continue
        raw_store_name = descs[0] if descs else ""
        if not raw_store_name:
            continue
        store_name = clean_result_name(raw_store_name)
        menu_name = ""
        for text in texts:
            if text == raw_store_name or text == store_name:
                continue
            if re.fullmatch(r"\d+\.\d", text):
                continue
            if re.fullmatch(r"\([\d,]+\)", text):
                continue
            if text in ["광고", "가게배달", "알뜰배달", "한집배달",
                        "픽업가능", "예약가능", ","]:
                continue
            if text.startswith("최소주문"):
                continue
            menu_name = text
            break
        menu_clean = menu_name.replace(" ", "").lower()
        similarity = SequenceMatcher(None, old_clean, menu_clean).ratio()
        review_count = 0
        for value in values:
            match = re.fullmatch(r"([\d,]+)개", value)
            if match:
                review_count = int(match.group(1).replace(",", ""))
                break
        delivery_fee = None
        fee_match = re.search(r"배달팁\s*([\d,]+)\s*원", raw_store_name)
        if fee_match and "할인" not in " ".join(values) and "할인" not in raw_store_name:
            delivery_fee = int(fee_match.group(1).replace(",", ""))
        if delivery_fee is None and "할인" not in " ".join(values) and "할인" not in raw_store_name:
            for i, value in enumerate(values):
                if value not in ["가게배달", "알뜰배달", "한집배달"]:
                    continue
                for fee_text in values[i + 1:i + 5]:
                    match = re.fullmatch(r"([\d,]+)(?:~([\d,]+))?원", fee_text.replace(" ", ""))
                    if match:
                        fee = match.group(2) or match.group(1)
                        delivery_fee = int(fee.replace(",", ""))
                        break
                if delivery_fee is not None:
                    break

        # 7. 점수
        old_brand = brand_name(old_store)
        new_brand = brand_name(store_name)
        old_brand_tokens = [re.sub(r"[^가-힣a-z0-9]", "", token.lower()) for token in old_brand.split()]
        new_brand_tokens = [re.sub(r"[^가-힣a-z0-9]", "", token.lower()) for token in new_brand.split()]
        old_brand_tokens = [token for token in old_brand_tokens if len(token) >= 3]
        new_brand_tokens = [token for token in new_brand_tokens if len(token) >= 3]
        brand_match = (old_brand == new_brand or any(token in new_brand_tokens for token in old_brand_tokens))
        brand_score = 10 if brand_match else 0
        menu_score = similarity * 30
        if delivery_fee is None:
            fee_score = 0
        elif delivery_fee <= 1000:
            fee_score = 20
        elif delivery_fee <= 2000:
            fee_score = 16
        elif delivery_fee <= 3000:
            fee_score = 12
        elif delivery_fee <= 4000:
            fee_score = 8
        elif delivery_fee <= 5000:
            fee_score = 4
        else:
            fee_score = 0
        if review_count >= 1000:
            review_score = 40
        elif review_count >= 500:
            review_score = 32
        elif review_count >= 200:
            review_score = 24
        elif review_count >= 100:
            review_score = 16
        elif review_count >= 50:
            review_score = 8
        elif review_count > 0:
            review_score = 4
        else:
            review_score = 0
        total_score = round(brand_score + menu_score + fee_score + review_score, 1)
        candidates.append({"store": store_name,
                           "menu": menu_name,
                           "brand_score": brand_score,
                           "menu_similarity": similarity,
                           "delivery_fee": delivery_fee,
                           "review_count": review_count,
                           "score": total_score})
    if sort_state == "SINGLE" and not candidates:
        raw_name, (cx1, cy1, cx2, cy2) = single_cards[0]
        store_name = clean_result_name(raw_name)
        texts = []
        values = []
        for node in root.iter():
            bounds = get_bounds(node)
            if not bounds:
                continue
            x1, y1, x2, y2 = bounds
            if cy1 <= y1 <= cy2 + 15 and x1 >= 180:
                t = node.attrib.get("text", "").strip()
                desc = node.attrib.get("content-desc", "").strip()
                if t:
                    texts.append((y1, t))
                    values.append(t)
                if desc:
                    values.append(desc)
        texts.sort()
        menu_name = next((t for _, t in texts
                          if not (t.startswith("최소주문") or t.startswith("배달팁")
                                  or t in ("할인", "픽업가능")
                                  or re.fullmatch(r"[\d.,()]+", t))), "")
        review_count = 0
        for value in values:
            m = re.fullmatch(r"([\d,]+)개", value)
            if m:
                review_count = int(m.group(1).replace(",", ""))
                break
        fee_match = next((re.search(r"배달팁\s*([\d,]+)원", v) for v in values if re.search(r"배달팁\s*([\d,]+)원", v)), None)
        delivery_fee = (int(fee_match.group(1).replace(",", "")) if fee_match and "할인" not in " ".join(values) else None)
        similarity = SequenceMatcher(None, old_menu.replace(" ", "").lower(),
                                     menu_name.replace(" ", "").lower()).ratio()
        old_brand = brand_name(old_store)
        new_brand = brand_name(store_name)
        old_tokens = [t for t in re.findall(r"[가-힣a-z0-9]+", old_brand) if len(t) >= 3]
        new_tokens = re.findall(r"[가-힣a-z0-9]+", new_brand)
        brand_match = old_brand == new_brand or any(t in new_tokens for t in old_tokens)
        brand_score = 10 if brand_match else 0
        menu_score = similarity * 30
        fee_score = (0 if delivery_fee is None else
                     20 if delivery_fee <= 1000 else 16 if delivery_fee <= 2000 else
                     12 if delivery_fee <= 3000 else 8 if delivery_fee <= 4000 else
                     4 if delivery_fee <= 5000 else 0)
        review_score = (40 if review_count >= 1000 else 32 if review_count >= 500 else
                        24 if review_count >= 200 else 16 if review_count >= 100 else
                        8 if review_count >= 50 else 4 if review_count > 0 else 0)
        candidates.append({"store": store_name, "menu": menu_name,
                           "brand_score": brand_score, "menu_similarity": similarity,
                           "delivery_fee": delivery_fee, "review_count": review_count,
                           "is_preparing": "준비중" in raw_name,
                           "score": round(brand_score + menu_score + fee_score + review_score, 1)})

    # 8. 추천
    candidates.sort(key=lambda x: x["score"], reverse=True)
    print("\n===== 신규 대상처 후보 =====")
    for i, item in enumerate(candidates, 1):
        print(f'\n{i}위: {item["store"]}')
        print("메뉴:", item["menu"])
        print("상호점수:", item["brand_score"])
        print("메뉴 유사도:", f'{item["menu_similarity"] * 100:.1f}%')
        print("배달비:", item["delivery_fee"])
        print("리뷰수:", item["review_count"])
        print("총점:", item["score"])
    food_terms = ("족발", "보쌈", "파스타", "필라프", "초밥", "쌀국수", "칼국수",
                  "짬뽕", "김밥", "피자", "치킨", "마라탕", "떡볶이", "햄버거",
                  "샐러드", "돈가스", "우동", "회", "커피")
    reference = " ".join([old_store, str(store.get("item") or ""), old_menu])
    expected_terms = [term for term in food_terms if term in reference]
    bakery_terms = ("파리바게뜨", "파리바게트", "뚜레쥬르", "베이커리", "제과점", "빵집")
    recommendable = []
    for item in candidates:
        if item.get("is_preparing"):
            continue
        name = item["store"]
        same_genre = bool(expected_terms and any(term in name for term in expected_terms))
        if "양식" in reference and any(word in name for word in bakery_terms):
            continue
        if item["brand_score"] > 0 or same_genre:
            recommendable.append(item)
    recommendation = recommendable[0] if recommendable else None
    if recommendation:
        print("\n★ 최종 추천:",
              recommendation["store"],
              "|",
              recommendation["score"],
              "점")
        return {"status": "REPLACEMENT_CANDIDATES",
                "candidates": candidates,
                "recommendation": recommendation,
                "order_sorted": sort_state != "SINGLE"}
    print("\n★ 자동 추천 가능한 같은 업종 대상처 없음 → 사람 확인")
    return {"status": "REPLACEMENT_REVIEW" if candidates else "NO_REPLACEMENT_CANDIDATE",
            "reason": "대체 대상처 업종·영업상태 확인 필요" if candidates else "대체 대상처 없음",
            "candidates": candidates,
            "recommendation": None,
            "order_sorted": sort_state != "SINGLE"}

## 3-7. Excel 비교 및 입력

In [ ]:
def money(value):
    if value is None or value == "":
        return None
    if isinstance(value, (int, float)):
        return int(value)
    value = str(value).replace(",", "").replace("원", "").strip()
    if value.isdigit():
        return int(value)
    return value

In [ ]:
def normalize_region_name(value):
    if value is None:
        return None
    text = str(value).strip()
    region_aliases = {
        "서울특별시": "서울특별시", "서울시": "서울특별시", "서울": "서울특별시",
        "부산광역시": "부산광역시", "부산시": "부산광역시", "부산": "부산광역시",
        "대구광역시": "대구광역시", "대구시": "대구광역시", "대구": "대구광역시",
        "인천광역시": "인천광역시", "인천시": "인천광역시", "인천": "인천광역시",
        "광주광역시": "광주광역시", "광주시": "광주광역시", "광주": "광주광역시",
        "대전광역시": "대전광역시", "대전시": "대전광역시", "대전": "대전광역시",
        "울산광역시": "울산광역시", "울산시": "울산광역시", "울산": "울산광역시",
        "세종특별자치시": "세종특별자치시", "세종시": "세종특별자치시", "세종": "세종특별자치시",
        "경기도": "경기도", "경기": "경기도",
        "강원특별자치도": "강원특별자치도", "강원도": "강원특별자치도", "강원": "강원특별자치도",
        "충청북도": "충청북도", "충북": "충청북도",
        "충청남도": "충청남도", "충남": "충청남도",
        "전북특별자치도": "전북특별자치도", "전라북도": "전북특별자치도", "전북": "전북특별자치도",
        "전라남도": "전라남도", "전남": "전라남도",
        "경상북도": "경상북도", "경북": "경상북도",
        "경상남도": "경상남도", "경남": "경상남도",
        "제주특별자치도": "제주특별자치도", "제주도": "제주특별자치도", "제주": "제주특별자치도"
    }
    for alias in sorted(region_aliases, key=len, reverse=True):
        if text.startswith(alias):
            text = region_aliases[alias] + text[len(alias):]
            break
    return text

In [ ]:
def clean_address(value):
    if value is None:
        return None
    value = normalize_region_name(value)
    value = re.sub(r"\s+", "", value)
    value = value.replace(",", "")
    return value

In [ ]:
def has_address_unit_conflict(old_address, new_address):
    if not old_address or not new_address:
        return False
    old_text = normalize_region_name(old_address)
    new_text = normalize_region_name(new_address)
    for unit in ("동", "층", "호"):
        old_units = set(re.findall(rf"(\d+)\s*{unit}(?![가-힣])", old_text))
        new_units = set(re.findall(rf"(\d+)\s*{unit}(?![가-힣])", new_text))
        if old_units and new_units and old_units != new_units:
            return True
    return False

def same_address(old_address, new_address):
    if old_address is None or new_address is None:
        return False
    if has_address_unit_conflict(old_address, new_address):
        return False
    old_text = normalize_region_name(old_address)
    new_text = normalize_region_name(new_address)
    old_text = re.sub(r"\s+", " ", str(old_text).replace(",", " ").strip())
    new_text = re.sub(r"\s+", " ", str(new_text).replace(",", " ").strip())
    if old_text == new_text:
        return True
    old_tokens = old_text.split()
    new_tokens = new_text.split()
    if len(new_tokens) >= len(old_tokens) and new_tokens[:len(old_tokens)] == old_tokens:
        return True
    if len(old_tokens) >= 5 and len(new_tokens) >= 5 and old_tokens[:5] == new_tokens[:5]:
        unit_conflict = False
        for unit in ["동", "층", "호"]:
            old_units = set(re.findall(rf"(\d+)\s*{unit}", old_text))
            new_units = set(re.findall(rf"(\d+)\s*{unit}", new_text))
            if old_units and new_units and old_units != new_units:
                unit_conflict = True
                break
        if not unit_conflict:
            return True
    old = clean_address(old_address)
    new = clean_address(new_address)
    if old is None or new is None:
        return False
    old_simple = re.sub(r"\([^,()]*?(?:동|읍|면|리|가)\)$", "", old)
    new_simple = re.sub(r"\([^,()]*?(?:동|읍|면|리|가)\)$", "", new)
    if old_simple == new_simple:
        return True
    old_base = re.sub(r"\([^()]*\)$", "", old)
    new_base = re.sub(r"\([^()]*\)$", "", new)
    if (old_base != old) != (new_base != new) and old_base == new_base:
        return True
    old_no_gu = re.sub(r"^((?:서울특별시|[가-힣]+광역시|세종특별자치시))[가-힣]+구", r"\1", old, count=1)
    new_no_gu = re.sub(r"^((?:서울특별시|[가-힣]+광역시|세종특별자치시))[가-힣]+구", r"\1", new, count=1)
    if old_no_gu != old and old_no_gu == new:
        return True
    if new_no_gu != new and new_no_gu == old:
        return True
    old_numbers = re.findall(r"\d+(?:-\d+)?", old)
    new_numbers = re.findall(r"\d+(?:-\d+)?", new)
    if old_numbers == new_numbers and (old in new or new in old):
        return True
    return False

In [ ]:
def address_similarity(old_address, new_address):
    old = clean_address(old_address)
    new = clean_address(new_address)
    if old is None or new is None:
        return 0.0
    return SequenceMatcher(None, old, new).ratio()

In [ ]:
def is_jibun_address_change(old_address, new_address):
    old_text = str(old_address or "")
    new_text = str(new_address or "")
    pattern = r"([가-힣]+동\d*가|[가-힣]+(?:동|리|읍|면))\s+(\d+(?:-\d+)?)"
    old_match = re.search(pattern, old_text)
    new_match = re.search(pattern, new_text)
    if not old_match or not new_match:
        return False
    old_area, old_lot = old_match.groups()
    new_area, new_lot = new_match.groups()
    return old_area == new_area and old_lot != new_lot

In [ ]:
def check_address(old_address, business_address, store_address, target_changed):
    if target_changed:
        new_address = store_address or business_address
        if new_address is None:
            return {"new_address": None, "review": True, "changed": False, "similarity": 0.0, "reason": "신규 대상처 주소 확인 불가"}
        return {"new_address": new_address,
                "review": False,
                "changed": not same_address(old_address, new_address),
                "similarity": address_similarity(old_address, new_address),
                "reason": ""}
    business_same = same_address(old_address, business_address)
    store_same = same_address(old_address, store_address)
    business_sim = address_similarity(old_address, business_address)
    store_sim = address_similarity(old_address, store_address)
    if not business_same and not store_same:
        if not has_address_unit_conflict(old_address, business_address) and is_jibun_address_change(old_address, business_address):
            return {"new_address": business_address,
                    "review": False,
                    "changed": True,
                    "similarity": business_sim,
                    "reason": ""}
        if not has_address_unit_conflict(old_address, store_address) and is_jibun_address_change(old_address, store_address):
            return {"new_address": store_address, "review": False, "changed": True, "similarity": store_sim, "reason": ""}
    if business_same:
        new_address = business_address
    elif store_same:
        new_address = store_address
    elif business_sim >= store_sim:
        new_address = business_address
    else:
        new_address = store_address
    similarity = max(business_sim, store_sim)
    unit_conflict = has_address_unit_conflict(old_address, new_address)
    review = unit_conflict or not (business_same or store_same or similarity >= 0.85)
    if unit_conflict:
        reason = "상세주소 동·층·호 불일치"
    elif not review:
        reason = ""
    elif similarity >= 0.80:
        reason = "동일주소 여부 확인"
    else:
        reason = "주소 불일치 확인"
    return {"new_address": new_address, "review": review, "changed": False, "similarity": similarity, "reason": reason}

In [ ]:
def find_address_typo(old_address, new_address):
    old_text = str(old_address or "")
    new_text = str(new_address or "")
    old_tokens = re.findall(r"[가-힣]+(?:동|읍|면|리|가)", old_text)
    new_tokens = re.findall(r"[가-힣]+(?:동|읍|면|리|가)", new_text)
    old_numbers = re.findall(r"\d+(?:-\d+)?", old_text)
    new_numbers = re.findall(r"\d+(?:-\d+)?", new_text)
    if old_numbers != new_numbers:
        return None
    for old_token in old_tokens:
        for new_token in new_tokens:
            if old_token == new_token:
                continue
            if len(old_token) != len(new_token):
                continue
            if old_token[-1] != new_token[-1]:
                continue
            diff_count = sum(a != b for a, b in zip(old_token, new_token))
            if diff_count == 1:
                return {"old": old_token, "new": new_token}
    return None

In [ ]:
def delivery_status(old_fee, new_fee, target_changed):
    if target_changed and old_fee != new_fee:
        return "대상처변동"
    if old_fee is None and new_fee is not None:
        return "신규"
    if old_fee is not None and new_fee is None:
        return "탈락"
    return ""

In [ ]:
def mark_red(cell):
    font = copy(cell.font)
    font.color = "FFFF0000"
    cell.font = font

In [ ]:
def write_review_note(ws, row, result):
    review_types = []
    if result is None:
        review_types.append("기술 실패")
    else:
        status = result.get("status")
        reason = result.get("reason", "")
        if status == "SAME_STORE_REVIEW":
            review_types.append("유사 대상처 확인")
        elif status == "MULTIPLE_MATCH_REVIEW":
            review_types.append("복수 대상처 확인")
        elif (
            result.get("target_changed")
            or status in ["REPLACEMENT_SEARCH_FAIL", "NO_REPLACEMENT_CANDIDATE"]):
            review_types.append("신규 대상처 확인")
        if result.get("menu_exists") is False:
            review_types.append("메뉴 확인")
        if result.get("address_review"):
            review_types.append("주소 확인")
        if result.get("review") and not review_types:
            review_types.append("기술 실패")
    review_types = list(dict.fromkeys(review_types))
    review_col = get_column_letter(column_index_from_string(RESULT_COLS["notes"]) + 1)
    ws[f"{review_col}{row}"] = ", ".join(review_types)

In [ ]:
def normalize_owner_name(value):
    if value is None:
        return ""
    text = unicodedata.normalize(
        "NFKC",
        str(value)
    ).casefold()
    return "".join(ch for ch in text if ch.isalnum())

In [ ]:
def compare_and_write(ws, store, delivery=None, information=None, menu_exists=None, new_store=None, review_types=None):
    row = store["row"]
    review_types = list(review_types or [])

    # 1. 전월값
    old_store = store["store"]
    old_owner = store["owner"]
    old_address = ws[f"K{row}"].value
    prev_store_min = money(ws[f'{PREV_COLS["store_min"]}{row}'].value)
    prev_store_fee = money(ws[f'{PREV_COLS["store_fee"]}{row}'].value)
    prev_saver_min = money(ws[f'{PREV_COLS["saver_min"]}{row}'].value)
    prev_saver_fee = money(ws[f'{PREV_COLS["saver_fee"]}{row}'].value)
    prev_one_min = money(ws[f'{PREV_COLS["one_min"]}{row}'].value)
    prev_one_fee = money(ws[f'{PREV_COLS["one_fee"]}{row}'].value)
    if new_store is None:
        new_store = old_store
    target_changed = old_store != new_store

    # 2. 배달정보
    delivery_ok = delivery is not None
    curr_store_min = None
    curr_store_fee = None
    curr_saver_min = None
    curr_saver_fee = None
    curr_one_min = None
    curr_one_fee = None
    if delivery_ok:
        curr_min = money(delivery["min_order"])
        curr_store_fee = money(delivery["store_fee"])
        curr_saver_fee = money(delivery["saver_fee"])
        curr_one_fee = money(delivery["one_fee"])
        curr_store_min = curr_min if curr_store_fee is not None else None
        curr_saver_min = curr_min if curr_saver_fee is not None else None
        curr_one_min = curr_min if curr_one_fee is not None else None

    # 3. 사업자정보
    information_ok = information is not None
    new_owner = None
    owner_changed = False
    new_address = None
    address_review = False
    address_changed = False
    address_similarity_value = 0.0
    address_review_reason = ""
    if information_ok:
        new_owner = information.get("owner")
        old_owner_clean = unicodedata.normalize("NFKC", str(old_owner or "")).casefold()
        new_owner_clean = unicodedata.normalize("NFKC", str(new_owner or "")).casefold()
        old_owner_clean = "".join(ch for ch in old_owner_clean if ch.isalnum())
        new_owner_clean = "".join(ch for ch in new_owner_clean if ch.isalnum())
        if not new_owner_clean:
            owner_changed = False
        elif not old_owner_clean:
            owner_changed = True
        elif old_owner_clean == new_owner_clean:
            owner_changed = False
        elif old_owner_clean in new_owner_clean or new_owner_clean in old_owner_clean:
            owner_changed = False
        else:
            owner_changed = True
        address_result = check_address(old_address,
                                       information.get("address"),
                                       information.get("store_address"),
                                       target_changed)
        new_address = address_result["new_address"]
        address_review = address_result["review"]
        address_changed = address_result["changed"]
        address_similarity_value = address_result["similarity"]
        address_review_reason = address_result["reason"]
        if address_review:
            review_types.append("주소 확인")

    # 4. 메뉴 확인
    if menu_exists is False:
        review_types.append("메뉴 확인")

    # 5. 대상처 변경
    if target_changed:
        review_types.append("신규 대상처 확인")
    review_types = list(dict.fromkeys(review_types))

    # 6. 배달정보 비교
    min_changed = False
    fee_changed = False
    if delivery_ok:
        min_changed = (
            (prev_store_fee is not None and curr_store_fee is not None and prev_store_min != curr_store_min)
            or (prev_saver_fee is not None and curr_saver_fee is not None and prev_saver_min != curr_saver_min)
            or (prev_one_fee is not None and curr_one_fee is not None and prev_one_min != curr_one_min)
        )
        fee_changed = (
            (prev_store_fee is not None and curr_store_fee is not None and prev_store_fee != curr_store_fee)
            or (prev_saver_fee is not None and curr_saver_fee is not None and prev_saver_fee != curr_saver_fee)
            or (prev_one_fee is not None and curr_one_fee is not None and prev_one_fee != curr_one_fee)
        )
    store_status = ""
    saver_status = ""
    one_status = ""
    if delivery_ok:
        store_status = delivery_status(prev_store_fee, curr_store_fee, target_changed)
        saver_status = delivery_status(prev_saver_fee, curr_saver_fee, target_changed)
        one_status = delivery_status(prev_one_fee, curr_one_fee, target_changed)

    # 7. 변동사항
    comparison_complete = delivery_ok and information_ok and not address_review
    change_type = ""
    if comparison_complete and fee_changed:
        if address_changed:
            change_type = "주소"
        elif min_changed:
            change_type = "품질규격"
        elif owner_changed:
            change_type = "기타"
        else:
            change_type = "가격"
    notes = []
    if not target_changed:
        if delivery_ok and min_changed:
            notes.append("배달최소주문금액 변경")
        if information_ok and owner_changed:
            notes.append(f"대표자 변경({old_owner or '-'} -> {new_owner or '-'})")
        if information_ok and not address_review and address_changed:
            notes.append("주소 변경")
    note_text = ", ".join(notes)

    # 8. Excel 부분입력
    written_parts = []
    if not target_changed:
        if delivery_ok:
            ws[f'{CURR_COLS["store_min"]}{row}'] = curr_store_min
            ws[f'{CURR_COLS["store_fee"]}{row}'] = curr_store_fee
            ws[f'{CURR_COLS["saver_min"]}{row}'] = curr_saver_min
            ws[f'{CURR_COLS["saver_fee"]}{row}'] = curr_saver_fee
            ws[f'{CURR_COLS["one_min"]}{row}'] = curr_one_min
            ws[f'{CURR_COLS["one_fee"]}{row}'] = curr_one_fee
            ws[f'{RESULT_COLS["store_status"]}{row}'] = store_status
            ws[f'{RESULT_COLS["saver_status"]}{row}'] = saver_status
            ws[f'{RESULT_COLS["one_status"]}{row}'] = one_status
            written_parts.append("배달정보")
            if prev_store_min != curr_store_min:
                mark_red(ws[f'{CURR_COLS["store_min"]}{row}'])
            if prev_store_fee != curr_store_fee:
                mark_red(ws[f'{CURR_COLS["store_fee"]}{row}'])
            if prev_saver_min != curr_saver_min:
                mark_red(ws[f'{CURR_COLS["saver_min"]}{row}'])
            if prev_saver_fee != curr_saver_fee:
                mark_red(ws[f'{CURR_COLS["saver_fee"]}{row}'])
            if prev_one_min != curr_one_min:
                mark_red(ws[f'{CURR_COLS["one_min"]}{row}'])
            if prev_one_fee != curr_one_fee:
                mark_red(ws[f'{CURR_COLS["one_fee"]}{row}'])
        if information_ok:
            if new_owner is not None:
                ws[f"J{row}"] = new_owner
                written_parts.append("대표자")
                if owner_changed:
                    mark_red(ws[f"J{row}"])
            if new_address is not None and not address_review:
                ws[f"K{row}"] = new_address
                written_parts.append("주소")
                if address_changed:
                    mark_red(ws[f"K{row}"])
        if comparison_complete:
            ws[f'{RESULT_COLS["change"]}{row}'] = change_type
            written_parts.append("변동사항")
        ws[f'{RESULT_COLS["notes"]}{row}'] = note_text
    if target_changed:
        print("엑셀 입력 보류: 신규 대상처 확인")
    elif review_types:
        print("엑셀 부분입력 완료:", ", ".join(written_parts) if written_parts else "입력값 없음")
        print("검토 필요:", ", ".join(review_types))
    else:
        print("엑셀 입력 완료")
    print("행:", row)
    if delivery_ok:
        print("가게배달:", curr_store_min, curr_store_fee, store_status)
        print("알뜰배달:", curr_saver_min, curr_saver_fee, saver_status)
        print("한집배달:", curr_one_min, curr_one_fee, one_status)
    print("최소주문금액 변동:", min_changed)
    print("배달비 가격변동:", fee_changed)
    print("변동사항:", change_type)
    print("비고:", note_text)
    if address_review:
        print("사람확인 필요:", address_review_reason)
        print("기존 주소:", old_address)
        print("현재 주소:", new_address)
        print("주소 유사도:", f"{address_similarity_value * 100:.1f}%")
    return {"notes": notes,
            "min_changed": min_changed,
            "fee_changed": fee_changed,
            "change_type": change_type,
            "address_review": address_review,
            "old_address": old_address,
            "new_address": new_address,
            "address_similarity": address_similarity_value,
            "address_review_reason": address_review_reason,
            "review_types": review_types,
            "excel_written_parts": written_parts,
            "comparison_complete": comparison_complete}

# 4. 대상처 조사

## 4-1. 대상처 1건 조사 (`survey_one`)

## 4-1. 개별 대상처 조사 (`survey_one`)

In [ ]:
def survey_one(store, ws):
    print("\n===== 조사 시작 =====")
    print("행:", store["row"])
    print("대상처:", store["store"])
    old_store = store["store"]
    new_store = old_store
    target_changed = False
    replacement_result = None

    # 1. 기존 대상처 검색
    open_result = open_store(old_store, store["item"])
    open_status = open_result["status"]
    store_name_typo = open_result.get("name_typo", False)
    store_name_excel = open_result.get("searched_name") or old_store
    store_name_baemin = open_result.get("selected")

    # 2. 기존 대상처 미발견 및 대체 탐색
    if open_status == "REPLACEMENT_NEEDED":
        print("기존 대상처 없음 → 신규 대상처 탐색")
        if not go_home():
            return {"status": "TECHNICAL_FAIL",
                    "review": True,
                    "row": store["row"],
                    "store": old_store,
                    "reason": "신규 대상처 탐색 전 첫 화면 복귀 실패",
                    "candidates": []}
        replacement_result = find_replacement_candidates(store)
        recommendation = replacement_result.get("recommendation")
        if recommendation is None:
            print("신규 대상처 추천 실패")
            return {"status": replacement_result.get("status", "REPLACEMENT_SEARCH_FAIL"),
                    "review": True,
                    "row": store["row"],
                    "store": old_store,
                    "reason": replacement_result.get("reason", "신규 대상처 추천 실패"),
                    "candidates": replacement_result.get("candidates", [])}
        new_store = recommendation["store"]
        print("★ 신규 대상처 추천:", new_store, "| 점수:", recommendation["score"])
        if not go_home():
            return {"status": "TECHNICAL_FAIL",
                    "review": True,
                    "row": store["row"],
                    "store": old_store,
                    "reason": "신규 대상처 진입 전 첫 화면 복귀 실패",
                    "candidates": []}
        new_open_result = open_store(new_store, store["item"])
        if new_open_result["status"] != "FOUND":
            return {"status": new_open_result["status"],
                    "review": True,
                    "row": store["row"],
                    "store": old_store,
                    "reason": "추천 신규 대상처 진입 실패: " + new_open_result["reason"],
                    "candidates": new_open_result.get("candidates", [])}
        new_store = new_open_result.get("selected") or new_store
        target_changed = True
        print("대상처 변경:", old_store, "→", new_store)

    # 3. 사람 확인 및 기술 오류 분기
    elif open_status != "FOUND":
        print("조사 보류:", open_result["reason"])
        return {"status": open_status,
                "review": True,
                "row": store["row"],
                "store": old_store,
                "reason": open_result["reason"],
                "candidates": open_result.get("candidates", [])}

    # 4. 배달정보
    delivery = get_delivery()
    if delivery is None:
        print("배달정보 수집 실패")
        return {"status": "PARTIAL_REVIEW",
                "review": True,
                "row": store["row"],
                "store": old_store,
                "reason": "배달정보 확인",
                "review_types": ["배달정보 확인"],
                "store_name_typo": store_name_typo,
                "store_name_excel": store_name_excel,
                "store_name_baemin": store_name_baemin,
                "candidates": []}
    if delivery["store_fee"] is None and delivery["saver_fee"] is None and delivery["one_fee"] is None:
        print("조사 보류: 배달비 3종 모두 확인 불가")
        return {"status": "PARTIAL_REVIEW",
                "review": True,
                "row": store["row"],
                "store": old_store,
                "reason": "배달정보 확인",
                "review_types": ["배달정보 확인"],
                "store_name_typo": store_name_typo,
                "store_name_excel": store_name_excel,
                "store_name_baemin": store_name_baemin,
                "candidates": []}

    # 5. 사업자정보
    information = get_information()
    if information is None:
        print("사업자정보 수집 실패")
        compare_result = compare_and_write(ws, store, delivery, None, None, new_store, ["사업자정보 확인"])
        return {"status": "PARTIAL_REVIEW",
                "review": True,
                "row": store["row"],
                "store": old_store,
                "target_changed": target_changed,
                "old_store": old_store,
                "new_store": new_store,
                "delivery": delivery,
                "information": None,
                "menu_exists": None,
                "reason": "사업자정보 확인",
                "review_types": compare_result["review_types"],
                "excel_written_parts": compare_result["excel_written_parts"],
                "store_name_typo": store_name_typo,
                "store_name_excel": store_name_excel,
                "store_name_baemin": store_name_baemin,
                "candidates": []}
    d.press("back")
    time.sleep(0.5)

    # 6. 기존 메뉴 확인
    if not open_menu_search(new_store):
        print("메뉴 검색창 진입 실패")
        compare_result = compare_and_write(ws, store, delivery, information, None, new_store, ["메뉴 확인"])
        return {"status": "PARTIAL_REVIEW",
                "review": True,
                "row": store["row"],
                "store": old_store,
                "target_changed": target_changed,
                "old_store": old_store,
                "new_store": new_store,
                "delivery": delivery,
                "information": information,
                "menu_exists": None,
                "reason": "메뉴 확인",
                "review_types": compare_result["review_types"],
                "excel_written_parts": compare_result["excel_written_parts"],
                "address_review": compare_result["address_review"],
                "old_address": compare_result["old_address"],
                "new_address": compare_result["new_address"],
                "address_similarity": compare_result["address_similarity"],
                "address_review_reason": compare_result["address_review_reason"],
                "store_name_typo": store_name_typo,
                "store_name_excel": store_name_excel,
                "store_name_baemin": store_name_baemin,
                "candidates": []}
    menu_exists, _ = search_menu(store["menu"], store["item"])
    d.press("back")
    time.sleep(0.5)

    # 7. 기존 메뉴 없으면 후보 추천
    menu_auto_match = False
    menu_auto_name = None
    menu_auto_similarity = 0.0
    menu_candidates = []
    menu_recommendation = None
    if menu_exists is False:
        menu_candidates, menu_recommendation = get_popular_menu_candidates(store["menu"])
        print("인기메뉴 후보:")
        for item in menu_candidates:
            print(f'{item["rank"]}위:', item["menu"], "| 유사도:", f'{item["similarity"] * 100:.1f}%')
        if menu_recommendation:
            print("★ 대체 추천:", menu_recommendation["menu"], "| 유사도:", f'{menu_recommendation["similarity"] * 100:.1f}%')
        else:
            print("★ 대체 추천 없음")
        if menu_recommendation and menu_recommendation["similarity"] >= 0.80:
            menu_auto_match = True
            menu_auto_name = menu_recommendation["menu"]
            menu_auto_similarity = menu_recommendation["similarity"]
            menu_exists = True
            print("※ 메뉴명 표기 차이 자동통과")
            print("기존:", store["menu"])
            print("현재:", menu_auto_name)
            print("유사도:", f"{menu_auto_similarity * 100:.1f}%")

    # 8. 엑셀 비교 / 입력
    compare_result = compare_and_write(ws, store, delivery, information, menu_exists, new_store)

    # 9. 주소 표기 비교
    address_auto_match = False
    if not target_changed and not compare_result["address_review"]:
        old_address_text = str(compare_result.get("old_address") or "").strip()
        new_address_text = str(compare_result.get("new_address") or "").strip()
        if old_address_text and new_address_text and old_address_text != new_address_text:
            address_auto_match = True
            print("※ 주소 표기 차이 자동통과")
            print("기존:", old_address_text)
            print("현재:", new_address_text)
            print("유사도:", f'{compare_result.get("address_similarity", 0.0) * 100:.1f}%')

    # 10. 결과
    result = {"status": "FOUND",
              "row": store["row"],
              "store": old_store,
              "target_changed": target_changed,
              "old_store": old_store,
              "new_store": new_store,
              "replacement_result": replacement_result,
              "delivery": delivery,
              "menu_exists": menu_exists,
              "menu_candidates": menu_candidates,
              "menu_recommendation": menu_recommendation,
              "menu_auto_match": menu_auto_match,
              "menu_auto_name": menu_auto_name,
              "menu_auto_similarity": menu_auto_similarity,
              "store_name_typo": store_name_typo,
              "store_name_excel": store_name_excel,
              "store_name_baemin": store_name_baemin,
              "address_auto_match": address_auto_match,
              "information": information,
              "notes": compare_result["notes"],
              "address_review": compare_result["address_review"],
              "old_address": compare_result["old_address"],
              "new_address": compare_result["new_address"],
              "address_similarity": compare_result["address_similarity"],
              "address_review_reason": compare_result["address_review_reason"],
              "review_types": compare_result["review_types"],
              "excel_written_parts": compare_result["excel_written_parts"],
              "comparison_complete": compare_result["comparison_complete"]}
    print("===== 조사 완료 =====")
    return result

## 4-2. 지역 단위 조사 (`survey_region`)

In [ ]:
def survey_region(SURVEY_REGION, show_header=True, overall_offset=0, overall_total=None):
    if SURVEY_REGION not in region_groups:
        raise ValueError(f"조사할 수 없는 지역입니다: {SURVEY_REGION}\n"
                         f"가능 지역: {sorted(region_groups.keys())}")
    region_addresses = region_groups[SURVEY_REGION]
    region_address_total = len(region_addresses)
    region_total = sum(len(stores) for stores in region_addresses.values())
    if overall_total is None:
        overall_total = region_total
    success = 0
    failed = 0
    processed = 0
    review_needed = []
    auto_passed = []
    replacement_queue = []
    survey_records = []
    if show_header:
        print("\n==============================")
        print("조사 지역:", SURVEY_REGION)
        print("주소지:", region_address_total, "개")
        print("조사 대상:", region_total, "개")
        print("==============================")
    for address_index, (address, stores) in enumerate(region_addresses.items(), 1):
        print("\n\n################################")
        print(f"[주소 {address_index}/{region_address_total}]", address)
        print("대상처:", len(stores), "개")
        print("################################")

        # 1. 주소 설정
        if not go_home():
            print("첫 화면 복귀 실패:", address)
            for store in stores:
                processed += 1
                failed += 1
                review_needed.append({"row": store["row"], "store": store["store"], "reason": "주소 설정 전 첫 화면 복귀 실패"})
            overall_processed = overall_offset + processed
            progress_pct = overall_processed / overall_total * 100 if overall_total else 0
            print(f"[전체 {overall_processed}/{overall_total} | {progress_pct:.1f}%] 지역 {SURVEY_REGION} {processed}/{region_total} | 주소 {address_index}/{region_address_total} | 성공 {success} | 확인/실패 {failed}")
            continue
        if not set_address(address):
            print("주소 설정 실패:", address)
            for store in stores:
                processed += 1
                failed += 1
                review_needed.append({"row": store["row"], "store": store["store"], "reason": "조사 주소 설정 실패"})
            overall_processed = overall_offset + processed
            progress_pct = overall_processed / overall_total * 100 if overall_total else 0
            print(f"[전체 {overall_processed}/{overall_total} | {progress_pct:.1f}%] 지역 {SURVEY_REGION} {processed}/{region_total} | 주소 {address_index}/{region_address_total} | 성공 {success} | 확인/실패 {failed}")
            continue

        # 2. 대상처 순차 조사
        previous_result = None
        for i, store in enumerate(stores):
            if i > 0:
                if previous_result is not None and previous_result.get("status") == "FOUND":
                    d.press("back")
                    search = d.xpath('//*[@clickable="true" and (contains(@text,"검색") or contains(@content-desc,"검색"))]')
                    if not search.wait(timeout=2):
                        print("검색창 확인 실패 → 기존 홈 복귀")
                        if not go_home():
                            print("첫 화면 복귀 실패")
                            break
                else:
                    print("직전 조사 미진입/보류 → back 생략, 홈에서 재검색")
                    if not go_home():
                        print("첫 화면 복귀 실패")
                        break
            result = survey_one(store, ws)
            previous_result = result
            write_review_note(ws, store["row"], result)
            survey_records.append({
                "log_id": f"{SURVEY_MONTH}_{SURVEY_REGION}_{store['row']}_{datetime.now():%Y%m%d%H%M%S%f}",
                "time": datetime.now(),
                "region": SURVEY_REGION,
                "address_index": address_index,
                "address": address,
                "store": store.copy(),
                "result": result,
                "progress": round((overall_offset + processed + 1) / overall_total * 100, 1)})
            processed += 1
            if result is not None:
                if result.get("store_name_typo"):
                    auto_passed.append({"row": store["row"],
                                        "store": store["store"],
                                        "reason": "대상처명 표기 차이 자동통과",
                                        "old_store_name": result.get("store_name_excel"),
                                        "new_store_name": result.get("store_name_baemin")})
                if result.get("menu_auto_match"):
                    auto_passed.append({"row": store["row"],
                                        "store": store["store"],
                                        "reason": "메뉴명 표기 차이 자동통과",
                                        "old_menu_name": store["menu"],
                                        "new_menu_name": result.get("menu_auto_name"),
                                        "menu_auto_similarity": result.get("menu_auto_similarity", 0.0)})
                if result.get("address_auto_match"):
                    auto_passed.append({"row": store["row"],
                                        "store": store["store"],
                                        "reason": "주소 표기 차이 자동통과",
                                        "old_address_text": result.get("old_address"),
                                        "new_address_text": result.get("new_address"),
                                        "address_auto_similarity": result.get("address_similarity", 0.0)})

            # 3. 조사 결과 판정
            if result is None:
                failed += 1
                review_needed.append({"row": store["row"], "store": store["store"], "reason": "기술적 조사 실패"})
                print("조사 실패:", store["store"])
            elif result.get("status") == "REPLACEMENT_NEEDED":
                failed += 1
                replacement_queue.append({"row": store["row"],
                                          "store": store["store"],
                                          "category": store["category"],
                                          "item": store["item"],
                                          "old_menu": store["menu"]})
                print("신규 대상처 탐색 필요:", store["store"])
            elif result.get("review"):
                failed += 1
                review_needed.append({"row": store["row"],
                                      "store": store["store"],
                                      "reason": result["reason"],
                                      "candidates": result.get("candidates", [])})
            else:
                needs_review = False
                if result.get("target_changed"):
                    needs_review = True
                    review_needed.append({"row": store["row"],
                                          "store": store["store"],
                                          "reason": "신규 대상처 확인",
                                          "change_reason": "기존 대상처 없음",
                                          "old_store": result["old_store"],
                                          "new_store": result["new_store"],
                                          "replacement_candidates": (result.get("replacement_result") or {}).get("candidates", []),
                                          "replacement_recommendation": (result.get("replacement_result") or {}).get("recommendation"),
                                          "menu_exists": result.get("menu_exists"),
                                          "old_menu": store["menu"],
                                          "menu_candidates": result.get("menu_candidates", []),
                                          "menu_recommendation": result.get("menu_recommendation"),
                                          "delivery": result.get("delivery", {}),
                                          "information": result.get("information", {}),
                                          "new_address": result.get("new_address")})
                if result.get("menu_exists") is False and not result.get("target_changed"):
                    needs_review = True
                    review_needed.append({"row": store["row"],
                                          "store": store["store"],
                                          "reason": "기존 메뉴 검색 안 됨",
                                          "category": ws[f'D{store["row"]}'].value,
                                          "menu_category": ws[f'E{store["row"]}'].value,
                                          "old_menu": store["menu"],
                                          "menu_candidates": result.get("menu_candidates", []),
                                          "menu_recommendation": result.get("menu_recommendation")})
                    print("조사 실패(메뉴 확인 필요):", store["store"])
                if result.get("address_review"):
                    needs_review = True
                    review_needed.append({"row": store["row"],
                                          "store": store["store"],
                                          "reason": "사업자주소 불일치 - " + result.get("address_review_reason", ""),
                                          "old_address": result["old_address"],
                                          "new_address": result["new_address"],
                                          "address_similarity": result.get("address_similarity", 0.0)})
                if needs_review:
                    failed += 1
                else:
                    success += 1
            overall_processed = overall_offset + processed
            progress_pct = overall_processed / overall_total * 100 if overall_total else 0
            print(f"[전체 {overall_processed}/{overall_total} | {progress_pct:.1f}%] 지역 {SURVEY_REGION} {processed}/{region_total} | 주소 {address_index}/{region_address_total} | 성공 {success} | 확인/실패 {failed}")

    # 4. 지역 조사 종료
    auto_passed_rows = len({item["row"] for item in auto_passed})
    print("\n==============================")
    print("지역 조사 종료")
    print("==============================")
    print("조사 지역:", SURVEY_REGION)
    print("주소지:", region_address_total)
    print("전체 대상:", region_total)
    print("처리:", processed)
    print("성공:", success)
    print("사람 확인/실패:", failed)
    print("------------------------------")
    print("자동통과:", auto_passed_rows)
    return {"region": SURVEY_REGION,
            "total": region_total,
            "processed": processed,
            "success": success,
            "failed": failed,
            "review_needed": review_needed,
            "auto_passed": auto_passed,
            "replacement_queue": replacement_queue,
            "records":survey_records}

## 4-3. 조사로그 저장·서식 함수

조사로그 열 순서, **글꼴 8pt**, 첫 행 필터 및 검토용 열 배치를 적용합니다.

In [ ]:
def format_candidates(candidates):
    lines = []
    for i, item in enumerate(candidates or [], 1):
        if isinstance(item, str):
            lines.append(f"{i}위: {item}")
            continue
        name = item.get("store") or item.get("menu") or item.get("name") or ""
        details = [f"{i}위: {name}"]
        for label, key in [
            ("메뉴", "menu"),
            ("점수", "score"),
            ("메뉴유사도", "menu_similarity"),
            ("배달비", "delivery_fee"),
            ("리뷰", "review_count"),
            ("유사도", "similarity")
        ]:
            value = item.get(key)
            if value is None or (key == "menu" and name == value):
                continue
            if key in ("similarity", "menu_similarity"):
                value = f"{value * 100:.1f}%"
            details.append(f"{label}: {value}")
        lines.append(" | ".join(details))
    return "\n".join(lines)[:32000]

In [ ]:
LOG_COLUMN_ORDER = ['지역', '주소번호', '엑셀행', '대상처', '조사판정', '확인사유', '검토 처리', '검토 메모', '상태 코드', '기존 메뉴', '메뉴 확인/대체', '신규 대상처 추천', '메뉴 추천', '자동통과 항목', '가게 최소주문금액', '가게배달비', '알뜰 최소주문금액', '알뜰배달비', '한집 최소주문금액', '한집배달비', '현재 대표자', '현재 사업자주소', '현재 가게주소', '조사주소', '대상처 검색 후보', '신규 대상처 후보 상세', '메뉴 후보 상세', '비고', '조사월', '기록일', '전체진행률(%)', '로그ID']
LOG_COLUMN_WIDTHS = {
    "지역": 10, "주소번호": 9, "엑셀행": 9, "대상처": 30,
    "조사판정": 19, "확인사유": 38,
    "검토 처리": 13, "검토 메모": 38, "상태 코드": 25,
    "기존 메뉴": 28, "메뉴 확인/대체": 30,
    "신규 대상처 추천": 31, "메뉴 추천": 29,
    "자동통과 항목": 24,
    "가게 최소주문금액": 18, "가게배달비": 14,
    "알뜰 최소주문금액": 18, "알뜰배달비": 14,
    "한집 최소주문금액": 18, "한집배달비": 14,
    "현재 대표자": 16, "현재 사업자주소": 44,
    "현재 가게주소": 44, "조사주소": 44,
    "대상처 검색 후보": 48, "신규 대상처 후보 상세": 52,
    "메뉴 후보 상세": 52, "비고": 38,
    "조사월": 10, "기록일": 21,
    "전체진행률(%)": 17, "로그ID": 38,
}
LOG_FONT = Font(name="맑은 고딕", size=8)
LOG_HEADER_FONT = Font(name="맑은 고딕", size=8, bold=True, color="FFFFFF")
LOG_HEADER_FILL = PatternFill(fill_type="solid", fgColor="30445F")

def format_log_row(sheet, row, headers):
    """새로 추가된 조사로그 행의 표시 형식만 설정."""
    sheet.row_dimensions[row].height = 19
    for col, title in enumerate(headers, 1):
        cell = sheet.cell(row, col)
        cell.font = LOG_FONT
        cell.alignment = Alignment(vertical="center")
        if title == "기록일" and cell.value is not None:
            cell.number_format = "yyyy-mm-dd hh:mm:ss"

def prepare_log_sheet(log_ws):
    """기존 로그의 열 이름을 기준으로 보존·재배치하고, 8pt·필터를 적용."""
    if getattr(log_ws, "_log_layout_ready", False):
        return {
            str(log_ws.cell(1, col).value): col
            for col in range(1, log_ws.max_column + 1)
            if log_ws.cell(1, col).value is not None
        }
    old_columns = {}
    for col in range(1, log_ws.max_column + 1):
        name = log_ws.cell(1, col).value
        if name is None or not str(name).strip():
            continue
        name = str(name).strip()
        if name in old_columns:
            raise ValueError(f"조사로그 열 제목 중복: {name}")
        old_columns[name] = col
    target_headers = LOG_COLUMN_ORDER + [name for name in old_columns if name not in LOG_COLUMN_ORDER]
    existing_headers = [
        str(log_ws.cell(1, col).value).strip()
        for col in range(1, log_ws.max_column + 1)
        if log_ws.cell(1, col).value is not None
        and str(log_ws.cell(1, col).value).strip()
    ]
    has_blank_header = len(old_columns) != log_ws.max_column
    needs_reorder = has_blank_header or existing_headers != target_headers
    if needs_reorder:
        existing_rows = [
            [log_ws.cell(row, old_columns[name]).value if name in old_columns else None for name in target_headers]
            for row in range(2, log_ws.max_row + 1)
        ]
        log_ws.delete_cols(1, log_ws.max_column)
        for col, title in enumerate(target_headers, 1):
            log_ws.cell(1, col, title)
        for row_idx, values in enumerate(existing_rows, 2):
            for col_idx, value in enumerate(values, 1):
                if value is not None:
                    log_ws.cell(row_idx, col_idx, value)
    for col, title in enumerate(target_headers, 1):
        cell = log_ws.cell(1, col)
        cell.font = LOG_HEADER_FONT
        cell.fill = LOG_HEADER_FILL
        cell.alignment = Alignment(vertical="center", horizontal="center")
        log_ws.column_dimensions[get_column_letter(col)].width = LOG_COLUMN_WIDTHS.get(title, 24)
    log_ws.row_dimensions[1].height = 24
    for row in range(2, log_ws.max_row + 1):
        format_log_row(log_ws, row, target_headers)
    log_ws.freeze_panes = "E2"
    log_ws.auto_filter.ref = f"A1:{get_column_letter(len(target_headers))}{max(log_ws.max_row, 1)}"
    log_ws.sheet_view.showGridLines = False
    log_ws._log_layout_ready = True
    return {title: col for col, title in enumerate(target_headers, 1)}

In [ ]:
def append_log_to_sheet(book=None,records=None):
    if book is None:
        book = wb
    if "조사로그" not in book.sheetnames:
        log_ws = book.create_sheet("조사로그")
    else:
        log_ws = book["조사로그"]
    if records is None :
        records = [record for region_result in all_results.values() for record in region_result.get("records", [])]
    if not records:
        prepare_log_sheet(log_ws)
        print("새 조사 기록 없음")
        return 0
    columns = prepare_log_sheet(log_ws)
    existing_ids = {log_ws.cell(row, columns["로그ID"]).value for row in range(2, log_ws.max_row + 1)}
    added = 0
    for record in records:
        log_id = record["log_id"]
        if log_id in existing_ids:
            continue
        store = record["store"]
        result = record.get("result") or {}
        replacement = result.get("replacement_result") or {}
        recommendation = replacement.get("recommendation") or {}
        delivery = result.get("delivery") or {}
        information = result.get("information") or {}
        menu_recommendation = result.get("menu_recommendation") or {}
        status = result.get("status", "NOT_PROCESSED")
        reasons = list(result.get("review_types") or [])
        if result.get("reason"):
            reasons.insert(0, result["reason"])
        if result.get("target_changed"):
            reasons.append("신규 대상처 확인")
        if result.get("menu_exists") is False:
            reasons.append("기존 메뉴 확인")
        if result.get("address_review"):
            reasons.append("주소 확인")
        reasons = list(dict.fromkeys(reasons))
        if status in ("TECHNICAL_FAIL", "NOT_PROCESSED"):
            judgement = "기술 실패/미조사"
        elif result.get("target_changed"):
            judgement = "신규 대상처 확인"
        elif status in (
            "MULTIPLE_MATCH_REVIEW",
            "SAME_STORE_REVIEW",
            "REPLACEMENT_NEEDED",
            "NO_REPLACEMENT_CANDIDATE",
            "REPLACEMENT_SEARCH_FAIL"
        ):
            judgement = "대상처 확인"
        elif result.get("review") or reasons:
            judgement = "확인 필요"
        else:
            judgement = "성공"
        auto = []
        if result.get("store_name_typo"):
            auto.append("대상처명 표기")
        if result.get("menu_auto_match"):
            auto.append("메뉴명 표기")
        if result.get("address_auto_match"):
            auto.append("주소 표기")
        min_order = delivery.get("min_order")
        candidates = replacement.get("candidates") or result.get("candidates") or []
        notes = result.get("notes") or ""
        if isinstance(notes, list):
            notes = ", ".join(notes)
        data = {
            "기록일": record["time"],
            "로그ID": log_id,
            "지역": record["region"],
            "주소번호": record["address_index"],
            "조사주소": record["address"],
            "엑셀행": store["row"],
            "대상처": store["store"],
            "조사판정": judgement,
            "확인사유": ", ".join(reasons),
            "자동통과 항목": ", ".join(auto),
            "메뉴 확인/대체": (
                result.get("menu_auto_name")
                or menu_recommendation.get("menu")
                or store.get("menu")
            ),
            "가게 최소주문금액": (
                min_order if delivery.get("store_fee") is not None else None
            ),
            "가게배달비": delivery.get("store_fee"),
            "알뜰 최소주문금액": (
                min_order if delivery.get("saver_fee") is not None else None
            ),
            "알뜰배달비": delivery.get("saver_fee"),
            "한집 최소주문금액": (
                min_order if delivery.get("one_fee") is not None else None
            ),
            "한집배달비": delivery.get("one_fee"),
            "비고": notes,
            "현재 대표자": information.get("owner"),
            "현재 사업자주소": information.get("address"),
            "현재 가게주소": information.get("store_address"),
            "전체진행률(%)": record.get("progress"),
            "기존 메뉴": store.get("menu"),
            "대상처 검색 후보": format_candidates(result.get("candidates")),
            "신규 대상처 추천": (
                recommendation.get("store")
                or (result.get("new_store") if result.get("target_changed") else "")
            ),
            "신규 대상처 후보 상세": format_candidates(candidates),
            "메뉴 추천": (
                menu_recommendation.get("menu")
                or result.get("menu_auto_name")
            ),
            "메뉴 후보 상세": format_candidates(result.get("menu_candidates")),
            "상태 코드": status,
            "조사월": SURVEY_MONTH
        }
        for title in data:
            if title not in columns:
                col = log_ws.max_column + 1
                log_ws.cell(1, col, title)
                columns[title] = col
        row_col = columns["엑셀행"]
        current_row = str(store["row"])
        for old_row in range(log_ws.max_row, 1, -1):
            if str(log_ws.cell(old_row, row_col).value) == current_row:
                existing_ids.discard(log_ws.cell(old_row, columns["로그ID"]).value)
                log_ws.delete_rows(old_row)
        target_row = log_ws.max_row + 1
        for title, value in data.items():
            log_ws.cell(target_row, columns[title], value)
        existing_ids.add(log_id)
        added += 1
        format_log_row(log_ws, target_row, [log_ws.cell(1, col).value for col in range(1, log_ws.max_column + 1)])
    log_ws.auto_filter.ref = (f"A1:{get_column_letter(log_ws.max_column)}{max(log_ws.max_row, 1)}")
    return added

## 4-4. BlueStacks 자동 재시작 함수

지역 조합 사이에 재시작합니다. 검증된 `Pie64` 인스턴스 설정을 사용합니다.

In [ ]:
BLUESTACKS_DIR = Path(r"C:\Program Files\BlueStacks_nxt")

def restart_bluestacks():
    global d
    player = BLUESTACKS_DIR / "HD-Player.exe"
    if os.name != "nt" or not player.exists():
        raise RuntimeError("Windows 또는 블루스택 실행 경로 확인 필요")
    command = (
        "$p = Get-CimInstance Win32_Process "
        "-Filter \"Name = 'HD-Player.exe'\"; "
        "$p | Select-Object ProcessId,CommandLine "
        "| ConvertTo-Json -Compress"
    )
    result = subprocess.run(["powershell", "-NoProfile", "-Command", command], capture_output=True, text=True, timeout=15)
    if result.returncode != 0:
        raise RuntimeError("블루스택 프로세스 확인 실패")
    processes = json.loads(result.stdout) if result.stdout.strip() else []
    if isinstance(processes, dict):
        processes = [processes]
    if len(processes) != 1:
        raise RuntimeError("블루스택 인스턴스를 정확히 1개 실행해야 합니다.")
    process = processes[0]
    instance = "Pie64"
    pid = process["ProcessId"]
    print("블루스택 인스턴스:", instance)
    print("프로세스 ID:", pid)
    serial = d.serial
    package = d.app_current()["package"]
    print("블루스택 종료:", instance)
    stopped = subprocess.run(["taskkill", "/F", "/PID", str(pid), "/T"], capture_output=True, text=True, timeout=20)
    if stopped.returncode != 0:
        raise RuntimeError("블루스택 종료 실패")
    time.sleep(3)
    print("블루스택 재실행")
    subprocess.Popen([str(player), "--instance", instance], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    deadline = time.monotonic() + 150
    while time.monotonic() < deadline:
        try:
            new_d = u2.connect(serial)
            new_d.info
            new_d.app_start(package)
            if new_d.app_current().get("package") == package:
                new_d.dump_hierarchy()
                d = new_d
                if go_home():
                    print("블루스택 재연결 완료")
                    return True
        except Exception:
            pass
        time.sleep(3)
    raise RuntimeError("블루스택 재연결 실패 - 다음 조사 중단")

## 4-5. 조사 보조 함수

선택 지역 로그 출력·값 보관 및 전체조사 주소지별 중간저장에 사용합니다.

In [ ]:
class LogTee:
    def __init__(self, screen, buffer):
        self.screen = screen
        self.buffer = buffer
    def write(self, text):
        self.screen.write(text)
        self.buffer.write(text)
    def flush(self):
        self.screen.flush()
        self.buffer.flush()

In [ ]:
def remember_region(region):
    for stores in region_groups[region].values():
        for store in stores:
            r = store["row"]
            SURVEY_SNAPSHOT[r] = {col: (ws[f"{col}{r}"].value, copy(ws[f"{col}{r}"].font)) for col in snapshot_cols}

In [ ]:
def save_survey_checkpoint(book, path):
    temp_file = path.with_name(path.stem + "_저장중.xlsx")
    book.save(temp_file)
    os.replace(temp_file, path)

# 5. 조사 실행

**전체조사(5-3)** 또는 **선택 지역(5-4)** 중 필요한 실행 방식을 사용하세요.

## 5-1. 조사 가능 지역 확인

In [ ]:
print("조사 가능 지역 개수:", len(region_groups))
for region in sorted(region_groups):
    address_count = len(region_groups[region])
    store_count = sum(len(stores) for stores in region_groups[region].values())
    print(region, "| 주소지:", address_count, "| 대상처:", store_count)

## 5-2. 추천 조사 조합

> 블루스택 과부하 방지를 위해 회차당 120~132개 대상처로 균일하게 분할

| 회차 | 지역 조합 | 주소지 | 대상처 |
|---:|---|---:|---:|
| 1 | **경기** | 11 | **132** |
| 2 | **서울 + 강원 + 인천** | 10 | **125** |
| 3 | **경남 + 전남 + 제주** | 10 | **120** |
| 4 | **경북 + 충북 + 세종** | 11 | **120** |
| 5 | **부산 + 광주 + 대구 + 울산** | 9 | **123** |
| 6 | **전북 + 충남 + 대전** | 10 | **125** |
| **합계** | **전체 지역** | **61** | **745** |

## 5-3. 전체 자동조사

`START_GROUP`, `END_GROUP`을 설정해 실행합니다. 6개 조합을 순차 조사하며 주소지마다 저장하고, 완료한 주소지는 재실행 시 건너뜁니다. **실행 즉시 실제 조사가 시작됩니다.**

In [ ]:
START_GROUP = 1
END_GROUP = 6
SURVEY_GROUPS = [
    ["경기"],
    ["서울", "강원", "인천"],
    ["경남", "전남", "제주"],
    ["경북", "충북", "세종"],
    ["부산", "광주", "대구", "울산"],
    ["전북", "충남", "대전"]
]
PROGRESS_SHEET = "자동조사진행"
TEMP_REGION_KEY = "__자동전체조사__"

# 1. 실행 전 검사
source = Path(OUTPUT_FILE)
RUN_FILE = source.with_name(source.stem + "_자동전체조사.xlsx")
if not source.exists():
    raise FileNotFoundError(f"원본 조사 파일이 없습니다: {source}")
if not 1 <= START_GROUP <= END_GROUP <= len(SURVEY_GROUPS):
    raise ValueError("START_GROUP / END_GROUP 설정 오류")
if TEMP_REGION_KEY in region_groups:
    raise RuntimeError("임시 조사 지역명이 이미 존재합니다. " "region_groups를 확인하세요.")
for group in SURVEY_GROUPS:
    for region in group:
        if region not in region_groups:
            raise ValueError(f"조사 지역 정보가 없습니다: {region}")
total = sum(len(stores) for group in SURVEY_GROUPS for region in group for stores in region_groups[region].values())
print("전체 자동 조사 준비")
print("전체 대상처:", total)
print("조사 조합:", START_GROUP, "~", END_GROUP)
print("결과 파일:", RUN_FILE)
if total != 745:
    raise RuntimeError(f"예상 대상처 745개와 실제 {total}개가 다릅니다. " "region_groups를 확인하세요.")

# 2. 결과 파일 준비
if not RUN_FILE.exists():
    copy2(source, RUN_FILE)
    print("\n새 전체 조사 파일 생성")
else:
    print("\n기존 전체 조사 파일에서 이어하기")
run_wb = load_workbook(RUN_FILE)
run_ws = run_wb[SHEET_NAME]
if PROGRESS_SHEET not in run_wb.sheetnames:
    progress_ws = run_wb.create_sheet(PROGRESS_SHEET)
    progress_ws.append(["지역", "주소지", "대상처수", "상태", "완료시각"])
else:
    progress_ws = run_wb[PROGRESS_SHEET]

# 3. 기존 완료 주소지 확인
done = {}
for values in progress_ws.iter_rows(
    min_row=2,
    values_only=True
):
    if not values or not values[0]:
        continue
    region, address, count, status = values[:4]
    if status == "DONE":
        key = (region, address)
        if region not in region_groups:
            raise RuntimeError(f"완료 기록에 없는 지역이 있습니다: {region}")
        if address not in region_groups[region]:
            raise RuntimeError(f"완료 기록에 없는 주소지가 있습니다: " f"{region} / {address}")
        expected = len(region_groups[region][address])
        if count != expected:
            raise RuntimeError(f"완료 기록의 대상처 수 불일치: " f"{region} / {address} " f"(기록 {count}, 현재 {expected})")
        if key in done:
            raise RuntimeError(f"완료 기록 중복: {region} / {address}")
        done[key] = count
completed = sum(done.values())
print("이전 완료 대상처:", completed)
print("남은 대상처:", total - completed)
original_wb = wb
original_ws = ws
wb = run_wb
ws = run_ws
start_time = time.monotonic()
newly_completed = 0
restart_count = 0
group_results = {}
try:

    # 4. 지역 조합 순회
    for group_no in range(
        START_GROUP,
        END_GROUP + 1
    ):
        regions = SURVEY_GROUPS[group_no - 1]
        pending = []
        for region in regions:
            for address, stores in region_groups[region].items():
                if (region, address) not in done:
                    pending.append((region, address, stores))
        if not pending:
            print(
                f"\n[조합 {group_no}/6] "
                "전체 완료된 조합 → 건너뜀"
            )
            continue
        group_total = sum(len(stores) for _, _, stores in pending)
        print(f"[조합 {group_no}/6] 조사 시작")
        print("지역:", ", ".join(regions))
        print("남은 주소지:", len(pending))
        print("남은 대상처:", group_total)
        print("전체 진행:", f"{completed}/{total}")

        # 5. 블루스택 재시작
        if group_no > 1:
            print("\n===== 블루스택 자동 재시작 =====")
            restart_start = time.monotonic()
            restart_bluestacks()
            restart_elapsed = (time.monotonic() - restart_start)
            restart_count += 1
            print(f"블루스택 재시작 완료: " f"{restart_elapsed:.1f}초")
        group_start = time.monotonic()
        group_processed = 0
        group_success = 0
        group_failed = 0

        # 6. 지역별 주소지 조사
        for region in regions:
            addresses = list(region_groups[region].items())
            for address_index, (address, stores) in enumerate(
                addresses,
                1
            ):
                if (region, address) in done:
                    print(
                        f"\n완료 주소지 건너뜀: "
                        f"{region} / {address}"
                    )
                    continue
                print("지역:", region)
                print("주소지:", f"{address_index}/{len(addresses)}")
                print("주소:", address)
                print("대상처:", len(stores))
                region_groups[TEMP_REGION_KEY] = {address: stores}
                try:
                    result = survey_region(TEMP_REGION_KEY, overall_offset=completed, overall_total=total)
                finally:
                    region_groups.pop(TEMP_REGION_KEY, None)

                # 7. 조사 결과 검증
                processed = result["processed"]
                records = result["records"]
                if (
                    processed != len(stores)
                    or len(records) != len(stores)
                ):
                    raise RuntimeError(
                        f"조사 건수 불일치: "
                        f"{region} / {address} "
                        f"(대상 {len(stores)}, "
                        f"처리 {processed}, "
                        f"로그 {len(records)})"
                    )
                technical_failures = [rec for rec in records if "TECHNICAL_FAIL" in str(rec)]
                if technical_failures:
                    print("\n기술 오류 발견")
                    print("지역:", region)
                    print("주소지:", address)
                    print("기술 오류 건수:", len(technical_failures))
                    raise RuntimeError(f"기술 오류 발생 - " f"주소지 미완료: {region} / {address}")

                # 8. 조사로그 기록
                for rec in records:
                    rec["region"] = region
                    rec["address_index"] = address_index
                added = append_log_to_sheet(book=run_wb, records=records)
                if added != len(stores):
                    raise RuntimeError(f"조사로그 기록 건수 불일치: " f"{region} / {address} " f"(예상 {len(stores)}, 기록 {added})")
                progress_ws.append([region, address, len(stores), "DONE", datetime.now().strftime("%Y-%m-%d %H:%M:%S")])

                # 9. 주소지별 엑셀 저장
                save_survey_checkpoint(run_wb, RUN_FILE)
                done[(region, address)] = len(stores)
                completed += len(stores)
                newly_completed += len(stores)
                group_processed += processed
                group_success += result["success"]
                group_failed += result["failed"]
                elapsed = (time.monotonic() - start_time)
                print("\n===== 주소지 저장 완료 =====")
                print("지역:", region)
                print("주소지:", address)
                print("처리:", processed)
                print("성공:", result["success"])
                print("확인/실패:", result["failed"])
                print("전체 진행:", f"{completed}/{total}")
                print("진행률:", f"{completed / total * 100:.1f}%")
                print(f"누적 소요: {elapsed / 60:.1f}분")
                print("엑셀 저장 완료")
        group_elapsed = (time.monotonic() - group_start)
        group_results[group_no] = {
            "regions": regions,
            "processed": group_processed,
            "success": group_success,
            "failed": group_failed,
            "elapsed": group_elapsed
        }
        print(f"조합 {group_no}/6 완료")
        print("지역:", ", ".join(regions))
        print("처리:", group_processed)
        print("성공:", group_success)
        print("확인/실패:", group_failed)
        print(f"조합 소요: {group_elapsed / 60:.1f}분")
        print("전체 진행:", f"{completed}/{total}")

    # 10. 전체 조사 종료
    total_elapsed = (time.monotonic() - start_time)
    print("전체 자동 조사 종료")
    print("전체 대상처:", total)
    print("누적 완료:", completed)
    print("이번 실행 조사:", newly_completed)
    print("남은 대상처:", total - completed)
    print("블루스택 재시작:", restart_count)
    print(f"이번 실행 소요: " f"{total_elapsed / 60:.1f}분")
    print("최종 결과 파일:", RUN_FILE)
    if completed == total:
        print("\n★ 745개 전체 조사 완료")
    else:
        print("\n※ 아직 미완료 대상처가 있습니다.")
finally:
    region_groups.pop(TEMP_REGION_KEY, None)
    wb = original_wb
    ws = original_ws
    run_wb.close()

---

## 5-4. 선택 지역 조사

특정 지역만 조사할 때 `SURVEY_REGIONS`를 설정하고 **5-4-1 조사 → 5-4-2 저장** 순서로 실행합니다. **전체조사(5-3)와 별개로 사용하세요.**

### 5-4-1. 선택 지역 조사 실행

In [ ]:
SURVEY_REGIONS = ["세종"]
SURVEY_SNAPSHOT = {}
invalid_regions = [r for r in SURVEY_REGIONS if r not in region_groups]
if invalid_regions:
    raise ValueError(f"조사할 수 없는 지역: {invalid_regions}")
review_col = get_column_letter(column_index_from_string(RESULT_COLS["notes"]) + 1)
snapshot_cols = ["J", "K", *CURR_COLS.values(), *RESULT_COLS.values(), review_col]
SURVEY_LOG = io.StringIO()
total_regions = len(SURVEY_REGIONS)
all_results = {}
if total_regions == 1:
    region = SURVEY_REGIONS[0]
    with redirect_stdout(LogTee(sys.stdout, SURVEY_LOG)):
        result = survey_region(region)
    all_results[region] = result
    remember_region(region)
else:
    total_addresses = sum(len(region_groups[r]) for r in SURVEY_REGIONS)
    total_targets = sum(sum(len(stores) for stores in region_groups[r].values()) for r in SURVEY_REGIONS)
    print("복수 지역 조사 시작")
    print("선택 지역:", ", ".join(SURVEY_REGIONS))
    print("지역 수:", total_regions, "개")
    print("전체 주소지:", total_addresses, "개")
    print("전체 조사 대상:", total_targets, "개")
    overall_offset = 0
    for region_index, region in enumerate(SURVEY_REGIONS, 1):
        print(f"\n\n▶ 지역 {region_index}/{total_regions}")
        with redirect_stdout(LogTee(sys.stdout, SURVEY_LOG)):
            result = survey_region(region, overall_offset=overall_offset, overall_total=total_targets)
        all_results[region] = result
        remember_region(region)
        overall_offset += result["processed"]
    total_processed = sum(x["processed"] for x in all_results.values())
    total_success = sum(x["success"] for x in all_results.values())
    total_failed = sum(x["failed"] for x in all_results.values())
    total_auto_passed = len({item["row"] for result in all_results.values() for item in result["auto_passed"]})
    print("복수 지역 조사 종료")
    print("선택 지역:", ", ".join(SURVEY_REGIONS))
    print("전체 조사 대상:", total_targets)
    print("처리:", total_processed)
    print("성공:", total_success)
    print("사람 확인/실패:", total_failed)
    print("자동통과:", total_auto_passed)

### 5-4-2. 선택 지역 조사 저장

In [ ]:
added = append_log_to_sheet()
wb.save(OUTPUT_FILE)
print("조사표 저장 완료:", OUTPUT_FILE)
print("조사로그 신규/갱신:", added, "건")
print("조사로그 전체:", wb["조사로그"].max_row - 1, "건")

# 6. 사람 확인 및 지정 재조사

결과 확인은 **실제 저장된 조사로그**를 기준으로 합니다.

## 6-1. 사람 확인 대상 조회

저장된 전체조사 파일을 우선 조회합니다. 선택 지역 파일을 보려면 `REVIEW_FILE`을 바꾸세요.

In [ ]:
DEFAULT_FULL_FILE = Path(OUTPUT_FILE).with_name(Path(OUTPUT_FILE).stem + "_자동전체조사.xlsx")
REVIEW_FILE = DEFAULT_FULL_FILE if DEFAULT_FULL_FILE.exists() else Path(OUTPUT_FILE)
if not REVIEW_FILE.exists():
    raise FileNotFoundError(f"검토할 엑셀 파일이 없습니다: {REVIEW_FILE}")
review_book = load_workbook(REVIEW_FILE, read_only=True, data_only=True)
try:
    if "조사로그" not in review_book.sheetnames:
        raise ValueError("조사로그 시트가 없습니다.")
    log_sheet = review_book["조사로그"]
    rows = log_sheet.iter_rows(values_only=True)
    headers = next(rows, None)
    if not headers:
        raise ValueError("조사로그에 열 제목이 없습니다.")
    columns = {str(name).strip(): i for i, name in enumerate(headers) if name is not None and str(name).strip()}
    needed = {"지역", "엑셀행", "대상처", "조사판정", "확인사유", "상태 코드"}
    if not needed.issubset(columns):
        raise ValueError(f"조사로그 필수 열 누락: {sorted(needed - set(columns))}")
    def field(row, title):
        i = columns.get(title)
        return row[i] if i is not None and i < len(row) else None
    to_review = []
    total_logs = 0
    for row in rows:
        if not any(value is not None for value in row):
            continue
        total_logs += 1
        verdict = str(field(row, "조사판정") or "").strip()
        status = str(field(row, "상태 코드") or "").strip()
        if ("확인" in verdict or "REVIEW" in status
                or status in {"TECHNICAL_FAIL", "REPLACEMENT_NEEDED"}):
            to_review.append(row)
    print("\n===== 사람 확인 필요 대상 =====")
    print("결과 파일:", REVIEW_FILE)
    print("전체 로그:", total_logs, "건")
    print("확인 대상:", len(to_review), "건")
    previous_region = None
    for row in sorted(to_review, key=lambda x: (
            str(field(x, "지역") or ""),
            str(field(x, "조사주소") or ""),
            int(field(x, "엑셀행") or 0))):
        region = str(field(row, "지역") or "미분류")
        if region != previous_region:
            print(f"\n[{region}]")
            previous_region = region
        print(f"{field(row, '엑셀행')}행 | {field(row, '대상처')} | " f"{field(row, '상태 코드') or '-'} | {field(row, '확인사유') or '-'}")
    if not to_review:
        print("사람 확인 필요 대상이 없습니다.")
finally:
    review_book.close()

## 6-2. 지정 대상처 재조사

**첫 줄 `TEST_ROWS`**에 조사할 엑셀 행 번호를 입력합니다. 여러 건이면 `[62, 486]`처럼 지정하세요.

In [ ]:
TEST_ROWS = [486]
full_path = Path(OUTPUT_FILE).with_name(Path(OUTPUT_FILE).stem + "_자동전체조사.xlsx")
ACTIVE_SURVEY_FILE = str(full_path) if full_path.exists() else OUTPUT_FILE
print("지정 재조사 대상 파일:", ACTIVE_SURVEY_FILE)
if isinstance(TEST_ROWS, int):
    TEST_ROWS = [TEST_ROWS]
test_wb = load_workbook(ACTIVE_SURVEY_FILE)
test_ws = test_wb["조사자료(rawdata)"]
print("지정 대상처 수정 조사")
print("조사 대상:", len(TEST_ROWS), "개")
print("행:", TEST_ROWS)
test_records = []
for test_index, TEST_ROW in enumerate(TEST_ROWS, 1):
    test_address = test_ws[f"H{TEST_ROW}"].value
    test_store = {"row": TEST_ROW,
                  "category": test_ws[f"D{TEST_ROW}"].value,
                  "item": test_ws[f"E{TEST_ROW}"].value,
                  "store": test_ws[f"I{TEST_ROW}"].value,
                  "owner": test_ws[f"J{TEST_ROW}"].value,
                  "business_address": test_ws[f"K{TEST_ROW}"].value,
                  "menu": test_ws[f"L{TEST_ROW}"].value}
    print(f"[{test_index}/{len(TEST_ROWS)}] 대상처 수정 조사")
    print("행:", TEST_ROW)
    print("주소:", test_address)
    print("대상처:", test_store["store"])
    print("기존 메뉴:", test_store["menu"])
    if not go_home():
        print("\n첫 화면 복귀 실패")
        continue
    if not set_address(test_address):
        print("\n주소 설정 실패")
        continue
    result = survey_one(test_store, test_ws)
    write_review_note(test_ws, TEST_ROW, result)
    test_records.append({
        "log_id": f"resurvey_{TEST_ROW}_{datetime.now():%Y%m%d%H%M%S%f}",
        "time": datetime.now(),
        "region": str(test_ws[f"F{TEST_ROW}"].value or "").strip(),
        "address_index": None,
        "address": test_address,
        "store": test_store.copy(),
        "result": result or {
            "status": "TECHNICAL_FAIL",
            "reason": "조사 결과 없음"},
        "progress": None})
    print("수정 조사 결과")
    if result is None:
        print("조사 실패")
        continue
    if result.get("status") != "FOUND":
        print("상태:", result.get("status"))
        print("사유:", result.get("reason", ""))
        candidates = result.get("candidates", [])
        if candidates:
            print("\n후보:")
            for i, item in enumerate(candidates, 1):
                if isinstance(item, dict):
                    print(f"{i}.", item.get("store", item))
                else:
                    print(f"{i}.", item)
        continue
    print("[대상처]")
    print("기존:", result.get("old_store"))
    print("현재:", result.get("new_store"))
    if result.get("target_changed"):
        print("→ 신규 대상처 확인 필요")
    else:
        print("→ 기존 대상처 유지")
    delivery = result.get("delivery") or {}
    print("\n[배달정보]")
    print("최소주문금액:", delivery.get("min_order"))
    print("가게배달:", delivery.get("store_fee"))
    print("알뜰배달:", delivery.get("saver_fee"))
    print("한집배달:", delivery.get("one_fee"))
    information = result.get("information") or {}
    print("\n[사업자정보]")
    print("대표자:", information.get("owner"))
    print("사업자명:", information.get("business_name"))
    print("사업자주소:", information.get("address"))
    print("가게주소:", information.get("store_address"))
    print("\n[메뉴]")
    print("기존 조사메뉴:", test_store["menu"])
    if result.get("menu_exists") is True:
        print("결과: 기존 메뉴 검색됨")
    else:
        print("결과: 기존 메뉴 검색 안 됨")
        menu_candidates = result.get("menu_candidates", [])
        if menu_candidates:
            print("\n대체 메뉴 후보:")
            for item in menu_candidates:
                print(f'{item["rank"]}위:', item["menu"], "|", f'{item["similarity"] * 100:.1f}%')
        recommendation = result.get("menu_recommendation")
        if recommendation:
            print("\n★ 메뉴 추천:", recommendation["menu"])
    replacement = result.get("replacement_result") or {}
    if result.get("target_changed"):
        print("\n[신규 대상처 후보]")
        for i, item in enumerate(replacement.get("candidates", []), 1):
            print(f'{i}위: {item["store"]} | 점수: {item["score"]} | 리뷰: {item["review_count"]} | 배달비: {item["delivery_fee"]}')
        recommendation = replacement.get("recommendation")
        if recommendation:
            print("\n★ 대상처 추천:", recommendation["store"])
    print("\n[주소 확인]")
    print("기존 주소:", result.get("old_address"))
    print("현재 주소:", result.get("new_address"))
    print("주소 유사도:", f'{result.get("address_similarity", 0) * 100:.1f}%')
    hold_reasons = []
    if result.get("target_changed"):
        hold_reasons.append("신규 대상처 확인")
    if result.get("menu_exists") is not True:
        hold_reasons.append("기존 메뉴 확인 필요")
    if result.get("address_review"):
        hold_reasons.append(result.get("address_review_reason") or "주소 확인 필요")
    if hold_reasons:
        print("엑셀 자동입력 보류")
        for reason in hold_reasons:
            print("-", reason)
    else:
        print("정상 자동입력 완료 대상")
print("지정 대상처 수정 조사 종료")
print("조사 대상:", len(TEST_ROWS), "개")

## 6-3. 지정 재조사 결과 저장

6-2의 조사 결과를 Excel에 반영합니다. 확인 후 실행하세요.

In [ ]:
if not test_records:
    raise ValueError("저장할 재조사 기록이 없습니다.")
latest_records = {int(rec["store"]["row"]): rec for rec in test_records}
rows = set(latest_records)
save_wb = load_workbook(ACTIVE_SURVEY_FILE)
save_ws = save_wb[SHEET_NAME]
review_col = get_column_letter(column_index_from_string(RESULT_COLS["notes"]) + 1)
cols = list(dict.fromkeys(["J", "K", *CURR_COLS.values(), *RESULT_COLS.values(), review_col]))
for r in rows:
    for col in cols:
        src = test_ws[f"{col}{r}"]
        dst = save_ws[f"{col}{r}"]
        dst.value = src.value
        dst._style = copy(src._style)
if "조사로그" in save_wb.sheetnames:
    log_ws = save_wb["조사로그"]
    headers = {cell.value: cell.column for cell in log_ws[1] if cell.value is not None}
    if "엑셀행" not in headers and log_ws.max_row > 1:
        raise ValueError("조사로그의 엑셀행 열을 찾지 못했습니다.")
    if "엑셀행" in headers:
        for i in range(log_ws.max_row, 1, -1):
            value = log_ws.cell(i, headers["엑셀행"]).value
            if str(value) in {str(r) for r in rows}:
                log_ws.delete_rows(i)
added = append_log_to_sheet(book=save_wb, records=list(latest_records.values()))
if added != len(latest_records):
    raise ValueError(f"조사로그 저장 실패: {added}/{len(latest_records)}")
save_wb.save(ACTIVE_SURVEY_FILE)
wb = save_wb
ws = save_ws
test_wb = save_wb
test_ws = save_ws
if "all_results" in globals():
    for region_result in all_results.values():
        if "records" in region_result:
            region_result["records"] = [rec for rec in region_result["records"] if int(rec["store"]["row"]) not in rows]
print("조사자료 + 조사로그 저장 완료")
print("저장 파일:", ACTIVE_SURVEY_FILE)
print("재조사 반영:", len(rows), "건")
print("조사로그 교체:", added, "건")
print("조사로그 전체:", save_wb["조사로그"].max_row - 1, "건")

## 6-4. 기존 조사로그 서식 정리 (선택)

열 순서·8pt 글꼴·필터를 적용하고 원본을 백업합니다.

In [ ]:
full_path = Path(OUTPUT_FILE).with_name(Path(OUTPUT_FILE).stem + "_자동전체조사.xlsx")
FORMAT_FILE = full_path if full_path.exists() else Path(OUTPUT_FILE)
if not FORMAT_FILE.exists():
    raise FileNotFoundError(f"서식 정리할 엑셀 파일 없음: {FORMAT_FILE}")
format_book = load_workbook(FORMAT_FILE)
try:
    if "조사로그" not in format_book.sheetnames:
        raise ValueError("이 파일에는 조사로그 시트가 없습니다.")
    log_sheet = format_book["조사로그"]
    before = log_sheet.max_row - 1
    prepare_log_sheet(log_sheet)
    after = log_sheet.max_row - 1
    if before != after:
        raise RuntimeError(f"로그 행 수가 변경됐습니다: {before} → {after}")
    backup = FORMAT_FILE.with_name(
        FORMAT_FILE.stem + "_서식수정전백업_"
        + datetime.now().strftime("%Y%m%d_%H%M%S") + FORMAT_FILE.suffix
    )
    copy2(FORMAT_FILE, backup)
    format_book.save(FORMAT_FILE)
    print("조사로그 열 배치·8pt·자동필터 적용:", FORMAT_FILE)
    print("보존된 로그:", after, "건")
    print("수정 전 백업:", backup)
finally:
    format_book.close()

# 부록 A. 오류 디버깅

화면 인식 오류나 XML 구조를 확인해야 할 때만 실행합니다. 일반 조사에는 필요하지 않습니다.

## A-1. 현재 화면 XML 확인 함수

In [ ]:
def inspect_current_screen():
    print("\n==============================")
    print("현재 화면 UI 구조 확인")
    print("==============================")
    xml = d.dump_hierarchy()
    root = ET.fromstring(xml)
    count = 0
    for node in root.iter():
        text = node.attrib.get("text", "").strip()
        desc = node.attrib.get("content-desc", "").strip()
        bounds = node.attrib.get("bounds", "")
        clickable = node.attrib.get("clickable", "")
        enabled = node.attrib.get("enabled", "")
        class_name = node.attrib.get("class", "")
        if not text and not desc:
            continue
        count += 1
        print(
            f"[{count}]",
            "| text:", repr(text),
            "| desc:", repr(desc),
            "| bounds:", bounds,
            "| clickable:", clickable,
            "| enabled:", enabled,
            "| class:", class_name
        )
    print("==============================")
    print("UI 구조 확인 끝")
    print("==============================")

## A-2. 현재 화면 구조 출력 — 선택 실행

In [ ]:
inspect_current_screen()